# Widget Delivery Case — Analysis

## 1. Cleaning

**What this section does:** loads the raw spreadsheet, profiles every column, removes the junk rows and the
phantom column, parses timestamps, and builds the 10 row-level derived fields defined in `CLAUDE.md`.

**What it feeds:** every later section (reconciliation, scoring, Q2/Q3, sweep, sensitivity) reads the cleaned
`df` built here. Output: `export_01_cleaned.xlsx`.

In [1]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 200)  # full tables in the PDF export (reconciliation table is 63 rows)
pd.set_option("display.max_colwidth", None)  # never truncate text cells (reconciliation notes)

def show(title, obj):
    """Display a titled table so every output reads without the code (PDF export hides code)."""
    display(Markdown(f"**{title}**"))
    display(obj)

RAW_PATH = "data/2026_Platform_Ops_Widget_Data.xlsx"
raw = pd.read_excel(RAW_PATH)
print("raw shape:", raw.shape)

raw shape: (1206, 39)


### 1.1 Junk rows and phantom column
The last 6 rows have no `MARKET_NAME`. Their only populated field is `COMPOSITE_SCORE` (the stray 34.5 plus two
other loose values). These look like summary cells sitting under the table, not deliveries.
`Unnamed: 38` is a column with no header. Its single value sits on row DELIVERY_NAME=930.

In [2]:
junk_rows = raw[raw["MARKET_NAME"].isna()]
show(f"Trailing rows with no MARKET_NAME ({len(junk_rows)} rows; only their non-empty columns shown)",
     junk_rows.dropna(axis=1, how="all"))
show("Non-null values in the phantom column 'Unnamed: 38'",
     raw.loc[raw["Unnamed: 38"].notna(), ["DELIVERY_NAME", "Unnamed: 38"]])

**Trailing rows with no MARKET_NAME (6 rows; only their non-empty columns shown)**

,COMPOSITE_SCORE
1200,NaN
1201,NaN
1202,NaN
1203,34.538563
1204,4.317320
1205,0.539665


**Non-null values in the phantom column 'Unnamed: 38'**

,DELIVERY_NAME,Unnamed: 38
0,930.0,4.431211


In [3]:
df = raw[raw["MARKET_NAME"].notna()].copy()
df = df.drop(columns=["Unnamed: 38"])
df = df.reset_index(drop=True)

show("After dropping junk rows and the phantom column", pd.DataFrame({"value": {
    "rows": len(df),
    "columns": df.shape[1],
    "Dashattan rows": int((df["MARKET_NAME"] == "Dashattan").sum()),
    "Doorlanta rows": int((df["MARKET_NAME"] == "Doorlanta").sum()),
    "DELIVERY_NAME unique": df["DELIVERY_NAME"].is_unique,
}}))

**After dropping junk rows and the phantom column**

,value
rows,1200
columns,38
Dashattan rows,600
Doorlanta rows,600
DELIVERY_NAME unique,True


### 1.2 Column profile (after dropping junk)
One row per column: type, missing count and %, distinct values, and range.

In [4]:
profile_rows = []
for col in df.columns:
    s = df[col]
    row = {
        "column": col,
        "dtype": str(s.dtype),
        "n_missing": s.isna().sum(),
        "pct_missing": round(100 * s.isna().mean(), 1),
        "n_unique": s.nunique(),
        "min": s.dropna().min() if s.notna().any() else None,
        "max": s.dropna().max() if s.notna().any() else None,
    }
    profile_rows.append(row)
column_profile = pd.DataFrame(profile_rows)
column_profile

,column,dtype,n_missing,pct_missing,n_unique,min,max
0,DELIVERY_NAME,float64,0,0.0,1200,1.0,1600.0
1,DELIVERY_RATING,float64,616,51.3,5,1.0,5.0
2,MARKET_NUMBER,float64,0,0.0,2,4001.0,4050.0
3,MARKET_NAME,str,0,0.0,2,Dashattan,Doorlanta
4,TIMEZONE,str,0,0.0,1,Galaxy A,Galaxy A
5,DASHER,str,0,0.0,54,Abhai,Zak
6,IS_FIRST_DELIVERY,float64,0,0.0,2,0.0,1.0
7,VEHICLE,str,0,0.0,3,bicycle,motorcycle
8,CREATED_AT,str,0,0.0,1200,2055-06-01T06:53:46.825653Z,2055-06-13T21:32:39.591796Z
9,QUOTED_DELIVERY_TIME,str,0,0.0,1188,2055-06-01T14:59:15.1Z,2055-06-13T22:45:00Z


In [5]:
# Missing counts by market, for columns that have any missing values
cols_with_missing = [c for c in df.columns if df[c].isna().any()]
df[cols_with_missing].isna().groupby(df["MARKET_NAME"]).sum().T

MARKET_NAME,Dashattan,Doorlanta
DELIVERY_RATING,333,283
DASHER_AT_STORE_TIME,42,82
CANCELLED_AT,600,600
DASHER_TO_RESTAURANT_DURATION,48,84
DASHER_WAIT_DURATION,88,103
RESTAURANT_TO_CUSTOMER_DURATION,0,1
DASHER_TO_STORE_TIME_CATEGORY,173,148
DASHER_AT_STORE_TIME_GALAXY_A,42,82
NUM_ON_TIME_DELIVERIES,415,76


In [6]:
# Value counts for the categorical / flag columns
for col in ["VEHICLE", "ASAP", "FRAUDULENT", "MANUALLY_ASSIGNED", "WAS_BATCHED",
            "IS_FIRST_DELIVERY", "DELIVERY_RATING", "TIMEZONE", "MARKET_NUMBER"]:
    print(col, df[col].value_counts(dropna=False).sort_index().to_dict())

print()
print("Dashers per market:", df.groupby("MARKET_NAME")["DASHER"].nunique().to_dict())

lifetime_cols = ["NUM_DELIVERIES", "COMPOSITE_STAR_RATING", "NUM_FIVE_STARS", "NUM_ON_TIME_DELIVERIES"]
varies = df.groupby(["MARKET_NAME", "DASHER"])[lifetime_cols].nunique(dropna=False) > 1
print("Dashers whose lifetime columns vary across rows:", varies.sum().to_dict())

vehicles_per_dasher = df.groupby(["MARKET_NAME", "DASHER"])["VEHICLE"].nunique()
print("Dashers with >1 vehicle:", vehicles_per_dasher[vehicles_per_dasher > 1].to_dict())

VEHICLE {'bicycle': 369, 'car': 828, 'motorcycle': 3}
ASAP {0.0: 74, 1.0: 1126}
FRAUDULENT {0.0: 1191, 1.0: 9}
MANUALLY_ASSIGNED {0.0: 1175, 1.0: 25}
WAS_BATCHED {0.0: 1043, 1.0: 157}
IS_FIRST_DELIVERY {0.0: 1193, 1.0: 7}
DELIVERY_RATING {1.0: 33, 2.0: 11, 3.0: 25, 4.0: 45, 5.0: 470, nan: 616}
TIMEZONE {'Galaxy A': 1200}
MARKET_NUMBER {4001.0: 600, 4050.0: 600}

Dashers per market: {'Dashattan': 24, 'Doorlanta': 30}
Dashers whose lifetime columns vary across rows: {'NUM_DELIVERIES': 0, 'COMPOSITE_STAR_RATING': 0, 'NUM_FIVE_STARS': 0, 'NUM_ON_TIME_DELIVERIES': 0}
Dashers with >1 vehicle: {('Doorlanta', 'Lauran'): 2}


### 1.3 Parse timestamps
All timestamps are ISO8601 strings. The UTC columns are used for arithmetic between timestamps. The `*_GALAXY_A`
columns hold local time (UTC−4) and are used only for hour-of-day. They also end in `Z`, but the check below shows
they sit exactly 4 hours behind UTC, so the `Z` is a mislabel. Reading `.dt.hour` from them gives the correct
local hour.

In [7]:
UTC_COLS = ["CREATED_AT", "QUOTED_DELIVERY_TIME", "ESTIMATED_DELIVERY_TIME", "ACTUAL_PICKUP_TIME",
            "ACTUAL_DELIVERY_TIME", "DASHER_ASSIGNED_TIME", "DASHER_CONFIRMED_TIME", "DASHER_AT_STORE_TIME"]
LOCAL_COLS = ["ACTUAL_PICKUP_TIME_GALAXY_A", "ACTUAL_DELIVERY_TIME_GALAXY_A",
              "DASHER_ASSIGNED_TIME_GALAXY_A", "DASHER_AT_STORE_TIME_GALAXY_A"]
TIMESTAMP_COLS = UTC_COLS + LOCAL_COLS

for col in TIMESTAMP_COLS:
    df[col] = pd.to_datetime(df[col], format="ISO8601")

# Local offset check: UTC minus local should be exactly 4 hours on every non-null row
for col in LOCAL_COLS:
    utc_col = col.replace("_GALAXY_A", "")
    offset_hours = (df[utc_col] - df[col]).dt.total_seconds() / 3600
    print(col, "UTC - local (hours):", offset_hours.value_counts().to_dict())

# ACTIVE_DATE is the local delivery date
local_delivery_date = df["ACTUAL_DELIVERY_TIME_GALAXY_A"].dt.tz_localize(None).dt.normalize()
print("ACTIVE_DATE != local delivery date:", (df["ACTIVE_DATE"] != local_delivery_date).sum())
print("ACTIVE_DATE range:", df["ACTIVE_DATE"].min().date(), "to", df["ACTIVE_DATE"].max().date())

ACTUAL_PICKUP_TIME_GALAXY_A UTC - local (hours): {4.0: 1200}
ACTUAL_DELIVERY_TIME_GALAXY_A UTC - local (hours): {4.0: 1200}
DASHER_ASSIGNED_TIME_GALAXY_A UTC - local (hours): {4.0: 1200}
DASHER_AT_STORE_TIME_GALAXY_A UTC - local (hours): {4.0: 1076}
ACTIVE_DATE != local delivery date: 0
ACTIVE_DATE range: 2055-06-01 to 2055-06-13


### 1.4 Integrity checks (report only; no rows changed)
Check that the timestamps run in order and that the `*_DURATION` columns (in seconds) agree with the timestamps.

In [8]:
order_pairs = [
    ("CREATED_AT", "DASHER_ASSIGNED_TIME"),
    ("DASHER_ASSIGNED_TIME", "DASHER_CONFIRMED_TIME"),
    ("DASHER_CONFIRMED_TIME", "DASHER_AT_STORE_TIME"),
    ("DASHER_AT_STORE_TIME", "ACTUAL_PICKUP_TIME"),
    ("ACTUAL_PICKUP_TIME", "ACTUAL_DELIVERY_TIME"),
    ("ESTIMATED_DELIVERY_TIME", "QUOTED_DELIVERY_TIME"),
]
order_check = pd.DataFrame(
    [{"earlier": a, "later": b, "rows_out_of_order": (df[a] > df[b]).sum()} for a, b in order_pairs]
)
display(order_check)

def seconds_between(start, end):
    return (df[end] - df[start]).dt.total_seconds()

duration_checks = {
    "CONFIRM_TO_DELIVER_DURATION": seconds_between("DASHER_CONFIRMED_TIME", "ACTUAL_DELIVERY_TIME"),
    "DASHER_TO_RESTAURANT_DURATION": seconds_between("DASHER_CONFIRMED_TIME", "DASHER_AT_STORE_TIME"),
    "DASHER_WAIT_DURATION": seconds_between("DASHER_AT_STORE_TIME", "ACTUAL_PICKUP_TIME"),
    "RESTAURANT_TO_CUSTOMER_DURATION": seconds_between("ACTUAL_PICKUP_TIME", "ACTUAL_DELIVERY_TIME"),
}
dur_rows = []
for col, from_ts in duration_checks.items():
    dur_rows.append({
        "duration_col": col,
        "missing": df[col].isna().sum(),
        "missing_but_timestamps_present": (df[col].isna() & from_ts.notna()).sum(),
        "of_which_negative_from_ts": (df[col].isna() & (from_ts < 0)).sum(),
        "disagree_gt_2s": ((df[col] - from_ts).abs() > 2).sum(),
    })
display(pd.DataFrame(dur_rows))

,earlier,later,rows_out_of_order
0,CREATED_AT,DASHER_ASSIGNED_TIME,0
1,DASHER_ASSIGNED_TIME,DASHER_CONFIRMED_TIME,0
2,DASHER_CONFIRMED_TIME,DASHER_AT_STORE_TIME,8
3,DASHER_AT_STORE_TIME,ACTUAL_PICKUP_TIME,67
4,ACTUAL_PICKUP_TIME,ACTUAL_DELIVERY_TIME,1
5,ESTIMATED_DELIVERY_TIME,QUOTED_DELIVERY_TIME,1


,duration_col,missing,missing_but_timestamps_present,of_which_negative_from_ts,disagree_gt_2s
0,CONFIRM_TO_DELIVER_DURATION,0,0,0,1
1,DASHER_TO_RESTAURANT_DURATION,132,8,8,0
2,DASHER_WAIT_DURATION,191,67,67,0
3,RESTAURANT_TO_CUSTOMER_DURATION,1,1,1,0


In [9]:
# The individual anomalous rows (single-row tables are transposed so the timestamps read down the page)
anomaly_cols = ["DELIVERY_NAME", "MARKET_NAME", "DASHER", "ASAP", "QUOTED_DELIVERY_TIME", "ESTIMATED_DELIVERY_TIME",
                "DASHER_CONFIRMED_TIME", "ACTUAL_PICKUP_TIME", "ACTUAL_DELIVERY_TIME",
                "CONFIRM_TO_DELIVER_DURATION", "RESTAURANT_TO_CUSTOMER_DURATION"]
show("Delivered before pickup",
     df.loc[df["ACTUAL_PICKUP_TIME"] > df["ACTUAL_DELIVERY_TIME"], anomaly_cols].T)
show("Estimate later than quote",
     df.loc[df["ESTIMATED_DELIVERY_TIME"] > df["QUOTED_DELIVERY_TIME"], anomaly_cols].T)
c2d_from_ts = seconds_between("DASHER_CONFIRMED_TIME", "ACTUAL_DELIVERY_TIME")
show("CONFIRM_TO_DELIVER_DURATION disagrees with timestamps (c2d_from_ts = seconds from the timestamps)",
     df.loc[(df["CONFIRM_TO_DELIVER_DURATION"] - c2d_from_ts).abs() > 2, anomaly_cols].assign(c2d_from_ts=c2d_from_ts).T)
show("IS_FIRST_DELIVERY == 1 (compare lifetime NUM_DELIVERIES)",
     df.loc[df["IS_FIRST_DELIVERY"] == 1, ["DELIVERY_NAME", "MARKET_NAME", "DASHER", "NUM_DELIVERIES"]])

**Delivered before pickup**

,949
DELIVERY_NAME,873.0
MARKET_NAME,Doorlanta
DASHER,Elon
ASAP,1.0
QUOTED_DELIVERY_TIME,2055-06-07 21:52:52.103000+00:00
ESTIMATED_DELIVERY_TIME,2055-06-07 21:32:20.103000+00:00
DASHER_CONFIRMED_TIME,2055-06-07 20:59:11.331781+00:00
ACTUAL_PICKUP_TIME,2055-06-07 21:29:15.715107+00:00
ACTUAL_DELIVERY_TIME,2055-06-07 21:28:23.637590+00:00
CONFIRM_TO_DELIVER_DURATION,1752.0


**Estimate later than quote**

,628
DELIVERY_NAME,200.0
MARKET_NAME,Doorlanta
DASHER,Jagar
ASAP,1.0
QUOTED_DELIVERY_TIME,2055-06-02 23:49:58.539460+00:00
ESTIMATED_DELIVERY_TIME,2055-06-02 23:50:58.539460+00:00
DASHER_CONFIRMED_TIME,2055-06-02 23:12:37.158532+00:00
ACTUAL_PICKUP_TIME,2055-06-02 23:18:50.154271+00:00
ACTUAL_DELIVERY_TIME,2055-06-02 23:29:34.358639+00:00
CONFIRM_TO_DELIVER_DURATION,1017.0


**CONFIRM_TO_DELIVER_DURATION disagrees with timestamps (c2d_from_ts = seconds from the timestamps)**

,785
DELIVERY_NAME,1339.0
MARKET_NAME,Dashattan
DASHER,KC
ASAP,1.0
QUOTED_DELIVERY_TIME,2055-06-12 00:07:51.648000+00:00
ESTIMATED_DELIVERY_TIME,2055-06-11 23:47:44.648000+00:00
DASHER_CONFIRMED_TIME,2055-06-11 23:22:31.669049+00:00
ACTUAL_PICKUP_TIME,2055-06-11 23:41:41.150079+00:00
ACTUAL_DELIVERY_TIME,2055-06-11 23:56:07.826347+00:00
CONFIRM_TO_DELIVER_DURATION,2055.0


**IS_FIRST_DELIVERY == 1 (compare lifetime NUM_DELIVERIES)**

,DELIVERY_NAME,MARKET_NAME,DASHER,NUM_DELIVERIES
10,613.0,Doorlanta,Kev,144.0
386,1220.0,Dashattan,Awstin,1403.0
513,729.0,Doorlanta,Jarry,198.0
610,118.0,Doorlanta,Matthew,223.0
618,1554.0,Dashattan,Dave,142.0
622,1111.0,Dashattan,Mike,736.0
1154,1141.0,Dashattan,Barry,418.0


### 1.5 Row-level derived fields
Definitions are copied from `CLAUDE.md` (metrics 1–10) and must not be varied. All timestamp arithmetic uses the
UTC columns; `hour_local` is the one field that reads a `*_GALAXY_A` (local, UTC−4) column.

**Source columns used**

| Raw column | Meaning | Unit |
|---|---|---|
| `CREATED_AT` | Order placed | UTC timestamp |
| `DASHER_ASSIGNED_TIME` | Dasher assigned to the order | UTC timestamp |
| `ESTIMATED_DELIVERY_TIME` | System's raw delivery estimate | UTC timestamp |
| `QUOTED_DELIVERY_TIME` | Delivery time promised to the customer | UTC timestamp |
| `ACTUAL_DELIVERY_TIME` | Delivery completed | UTC timestamp |
| `ACTUAL_DELIVERY_TIME_GALAXY_A` | Same instant, local time | local timestamp |
| `DASHER_AT_STORE_TIME` | Dasher tapped "arrived at store" (blank on 124 rows) | UTC timestamp |
| `DASHER_WAIT_DURATION` | Time from store arrival to pickup (blank on 191 rows) | **seconds** |
| `ASAP` | 1 = on-demand order, 0 = scheduled order | flag |

**Derived fields**

| # | Field | Definition | Formula | Unit / type | Notes |
|---|---|---|---|---|---|
| 1 | `late_min` | Minutes delivered after the quoted time (negative = early) | `(ACTUAL_DELIVERY_TIME − QUOTED_DELIVERY_TIME) / 60 s` | minutes | Measured against the **quoted** promise, never the estimate |
| 1 | `is_late` | Raw late flag | `late_min > 0` | bool | Strict: exactly on time is not late |
| 2 | `wait_min` | Merchant wait at the store | `DASHER_WAIT_DURATION / 60` | minutes | Missing stays NaN (15.9% of rows). Never imputed |
| 3 | `excess_wait` | Wait beyond the 10-minute merchant allowance | `max(wait_min − 10, 0)`; NaN → 0 | minutes | The **only** fillna in the analysis: missing wait = zero forgiveness |
| 4 | `late_min_adj` | Lateness after forgiving excess merchant wait | `late_min − excess_wait` | minutes | Attributes long merchant holds to the merchant, not the Dasher |
| 4 | `is_late_adj` | Adjusted late flag | `late_min_adj > 0` | bool | Basis of `ontime_adj` in the Dasher score |
| 5 | `severe` | Severe lateness | `late_min_adj > 15` | bool | On the **adjusted** number. Basis of `severe_rate` in the score |
| 6 | `assign_lag_min` | Dispatch delay: order created → Dasher assigned | `(DASHER_ASSIGNED_TIME − CREATED_AT) / 60 s` | minutes | Computed on all rows; meaningful for ASAP only (scheduled orders are assigned ahead of time by design) |
| 7 | `buffer_min` | Cushion the quote adds on top of the raw estimate | `(QUOTED_DELIVERY_TIME − ESTIMATED_DELIVERY_TIME) / 60 s` | minutes | Expect 0 for scheduled, ~17 for ASAP |
| 8 | `late_vs_estimate` | Late against the raw estimate instead of the quote | `ACTUAL_DELIVERY_TIME > ESTIMATED_DELIVERY_TIME` | bool | Used only to show the quote is the fair benchmark (ASAP ≈ 56%) |
| 9 | `hour_local` | Local hour of delivery, 0–23 | `ACTUAL_DELIVERY_TIME_GALAXY_A.dt.hour` | int | For hour-of-day analysis (Q2 demand peaks) |
| 10 | `arrival_logged` | Dasher logged a store arrival | `DASHER_AT_STORE_TIME.notna()` | bool | Basis of `arrival_share` (professionalism) in the score |

Fields 1–5 form a chain: `late_min` → subtract `excess_wait` → `late_min_adj` → thresholds at 0 (`is_late_adj`) and
15 (`severe`). The rest are independent of each other.

In [10]:
def minutes_between(start, end):
    return (df[end] - df[start]).dt.total_seconds() / 60

# 1. Lateness vs the quoted time (strictly greater than 0 = late)
df["late_min"] = minutes_between("QUOTED_DELIVERY_TIME", "ACTUAL_DELIVERY_TIME")
df["is_late"] = df["late_min"] > 0

# 2. Merchant wait in minutes (missing stays missing)
df["wait_min"] = df["DASHER_WAIT_DURATION"] / 60

# 3. Wait beyond 10 minutes is forgiven; missing wait -> 0 forgiveness
df["excess_wait"] = (df["wait_min"] - 10).clip(lower=0).fillna(0)

# 4. Adjusted lateness
df["late_min_adj"] = df["late_min"] - df["excess_wait"]
df["is_late_adj"] = df["late_min_adj"] > 0

# 5. Severe lateness (on the adjusted number)
df["severe"] = df["late_min_adj"] > 15

# 6. Assignment lag (meaningful for ASAP orders only)
df["assign_lag_min"] = minutes_between("CREATED_AT", "DASHER_ASSIGNED_TIME")

# 7. Quote buffer: quoted minus estimated
df["buffer_min"] = minutes_between("ESTIMATED_DELIVERY_TIME", "QUOTED_DELIVERY_TIME")

# 8. Late against the raw estimate
df["late_vs_estimate"] = df["ACTUAL_DELIVERY_TIME"] > df["ESTIMATED_DELIVERY_TIME"]

# 9. Local hour of delivery
df["hour_local"] = df["ACTUAL_DELIVERY_TIME_GALAXY_A"].dt.hour

# 10. Did the Dasher log arrival at the store?
df["arrival_logged"] = df["DASHER_AT_STORE_TIME"].notna()

DERIVED = ["late_min", "is_late", "wait_min", "excess_wait", "late_min_adj", "is_late_adj", "severe",
           "assign_lag_min", "buffer_min", "late_vs_estimate", "hour_local", "arrival_logged"]
df[DERIVED].describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
late_min,1200.0,NaN,NaN,NaN,-12.6152,18.222834,-61.879956,-24.375999,-14.813403,-3.961245,179.426294
is_late,1200,2,False,990,NaN,NaN,NaN,NaN,NaN,NaN,NaN
wait_min,1009.0,NaN,NaN,NaN,10.036455,9.729333,0.016667,3.466667,6.966667,13.733333,76.45
excess_wait,1200.0,NaN,NaN,NaN,2.949917,7.177276,0.0,0.0,0.0,1.633333,66.45
late_min_adj,1200.0,NaN,NaN,NaN,-15.565117,17.838054,-61.879956,-26.663741,-17.836475,-7.128004,179.426294
is_late_adj,1200,2,False,1039,NaN,NaN,NaN,NaN,NaN,NaN,NaN
severe,1200,2,False,1149,NaN,NaN,NaN,NaN,NaN,NaN,NaN
assign_lag_min,1200.0,NaN,NaN,NaN,20.104292,178.977265,0.044579,0.362908,4.637724,9.544172,5666.781894
buffer_min,1200.0,NaN,NaN,NaN,16.310097,7.343573,-1.0,10.383333,18.883333,20.725,33.55
late_vs_estimate,1200,2,True,646,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
# Sanity checks against CLAUDE.md's stated expectations
print("wait_min missing share:", round(df["wait_min"].isna().mean(), 3), "(expect ~16%)")
print("buffer_min mean by ASAP:", df.groupby("ASAP")["buffer_min"].mean().round(2).to_dict(),
      "(expect 0 scheduled, ~17 ASAP)")
print("late_vs_estimate mean, ASAP only:", round(df.loc[df["ASAP"] == 1, "late_vs_estimate"].mean(), 3),
      "(expect ~0.56)")

wait_min missing share: 0.159 (expect ~16%)
buffer_min mean by ASAP: {0.0: 0.0, 1.0: 17.38} (expect 0 scheduled, ~17 ASAP)
late_vs_estimate mean, ASAP only: 0.556 (expect ~0.56)


### 1.6 Hand-check: DELIVERY_NAME = 930
This row is scheduled (ASAP=0), has a missing wait, and has quoted == estimated. Its values are read straight from
the workbook with openpyxl, independently of pandas, and the derived fields are recomputed by hand.

In [12]:
wb = load_workbook(RAW_PATH, read_only=True)
ws = wb.active
header = [cell.value for cell in next(ws.iter_rows(min_row=1, max_row=1))]
raw_930 = None
for row in ws.iter_rows(min_row=2, values_only=True):
    if row[header.index("DELIVERY_NAME")] == 930:
        raw_930 = dict(zip(header, row))
        break
wb.close()

check_fields = ["ASAP", "DASHER", "MARKET_NAME", "QUOTED_DELIVERY_TIME", "ESTIMATED_DELIVERY_TIME",
                "ACTUAL_DELIVERY_TIME", "CREATED_AT", "DASHER_ASSIGNED_TIME", "DASHER_WAIT_DURATION",
                "DASHER_AT_STORE_TIME", "ACTUAL_DELIVERY_TIME_GALAXY_A"]
{f: raw_930[f] for f in check_fields}

{'ASAP': 0,
 'DASHER': 'Katelynn',
 'MARKET_NAME': 'Doorlanta',
 'QUOTED_DELIVERY_TIME': '2055-06-08T16:30:00Z',
 'ESTIMATED_DELIVERY_TIME': '2055-06-08T16:30:00Z',
 'ACTUAL_DELIVERY_TIME': '2055-06-08T16:43:26.092195Z',
 'CREATED_AT': '2055-06-08T14:40:31.776158Z',
 'DASHER_ASSIGNED_TIME': '2055-06-08T15:24:32.128072Z',
 'DASHER_WAIT_DURATION': None,
 'DASHER_AT_STORE_TIME': None,
 'ACTUAL_DELIVERY_TIME_GALAXY_A': '2055-06-08T12:43:26.092195Z'}

In [13]:
# Hand calculation from the raw strings above:
#   quoted   = 16:30:00 UTC, estimated = 16:30:00 UTC  -> buffer 0, scheduled order
#   actual   = 16:43:26.09 UTC                         -> late by 13.43 min
#   wait     = missing                                 -> excess_wait 0, adjusted lateness = 13.43 min
#   13.43 is not > 15                                  -> not severe
#   created 14:40:31.78, assigned 15:24:32.13          -> assign lag 44.01 min
#   local delivery 12:43                               -> hour_local 12
#   DASHER_AT_STORE_TIME missing                       -> arrival_logged False
row_930 = df.loc[df["DELIVERY_NAME"] == 930].iloc[0]
expected_930 = {
    "ASAP": 0, "late_min": 13.435, "is_late": True, "wait_min": np.nan, "excess_wait": 0.0,
    "late_min_adj": 13.435, "is_late_adj": True, "severe": False, "assign_lag_min": 44.006,
    "buffer_min": 0.0, "late_vs_estimate": True, "hour_local": 12, "arrival_logged": False,
}
check_930 = pd.DataFrame({
    "hand_calc": pd.Series(expected_930),
    "pipeline": row_930[list(expected_930)],
})
check_930["match"] = [
    (pd.isna(h) and pd.isna(p)) or (isinstance(h, float) and abs(h - p) < 0.01) or (h == p)
    for h, p in zip(check_930["hand_calc"], check_930["pipeline"])
]
display(check_930)
assert check_930["match"].all(), "row 930 hand-check failed"
print("DELIVERY_NAME 930 hand-check: PASS")

,hand_calc,pipeline,match
ASAP,0,0.0,True
late_min,13.435,13.43487,True
is_late,True,True,True
wait_min,NaN,NaN,True
excess_wait,0.0,0.0,True
late_min_adj,13.435,13.43487,True
is_late_adj,True,True,True
severe,False,False,True
assign_lag_min,44.006,44.005865,True
buffer_min,0.0,0.0,True


DELIVERY_NAME 930 hand-check: PASS


### 1.7 Acceptance checks

In [14]:
assert len(df) == 1200, len(df)
assert (df["MARKET_NAME"].value_counts() == 600).all()
assert not any(c.startswith("Unnamed") for c in df.columns)
REQUIRED_DERIVED = ["late_min", "wait_min", "excess_wait", "late_min_adj", "is_late_adj", "severe",
                    "assign_lag_min", "buffer_min", "late_vs_estimate", "hour_local", "arrival_logged"]
missing_derived = [c for c in REQUIRED_DERIVED if c not in df.columns]
assert not missing_derived, missing_derived
print("Phase 1 acceptance: PASS —", len(df), "rows,", df.shape[1], "columns")

Phase 1 acceptance: PASS — 1200 rows, 50 columns


### 1.8 Cleaning log
| # | Decision | Rows / cols affected | Basis |
|---|---|---|---|
| 1 | Dropped 6 trailing rows with no `MARKET_NAME`. Their only data is `COMPOSITE_SCORE` (34.54, 4.32, 0.54). They are loose cells, not deliveries | 6 rows | CLAUDE.md dataset facts |
| 2 | Dropped `Unnamed: 38`, a column with no header. Its only value is 4.43 on DELIVERY_NAME 930 | 1 col | CLAUDE.md dataset facts |
| 3 | Parsed 12 timestamp columns as ISO8601. UTC columns are used for arithmetic; the `*_GALAXY_A` columns, verified as UTC−4 on every row, are used for hour-of-day. Their `Z` suffix is a mislabel | 12 cols | CLAUDE.md dataset facts; offset check §1.3 |
| 4 | Converted durations from seconds to minutes when deriving (`wait_min`). The raw `*_DURATION` columns stay in seconds | — | CLAUDE.md |
| 5 | `DASHER_WAIT_DURATION` missing on 191 rows (15.9%) and kept as NaN. 124 of them have no store arrival logged. The other 67 have `DASHER_AT_STORE_TIME` *after* pickup (by 0–13 min), so the source left the wait blank instead of storing a negative | 191 rows | CLAUDE.md: never impute |
| 6 | `excess_wait` fills NaN wait with 0 (missing wait = zero forgiveness). This is the only fillna | 191 rows | CLAUDE.md metric 3 |
| 6a | The 67 late-logged arrivals (store arrival stamped after pickup) get zero wait forgiveness, because their wait is blank, but they still count as `arrival_logged = True` for the Professionalism component, because `DASHER_AT_STORE_TIME` is present | 67 rows | spec as written; visibility table in §2.3 |
| 7 | Kept DELIVERY_NAME 873 (Doorlanta, Elon). Its delivery timestamp is 52 s *before* pickup, so `RESTAURANT_TO_CUSTOMER_DURATION` is NaN. The metrics in scope don't use pickup→delivery, so nothing changes | 1 row | report only |
| 8 | Kept DELIVERY_NAME 200 (Doorlanta, Jagar, ASAP). Its estimate is 1 min later than the quote, giving buffer_min = −1.0 | 1 row | report only |
| 9 | Kept DELIVERY_NAME 1339 (Dashattan, KC). `CONFIRM_TO_DELIVER_DURATION` = 2055 s, but the timestamps give 2016 s (a 39 s gap). Metrics use timestamps, not this column | 1 row | report only |
| 10 | 8 rows have store arrival before confirmation, so `DASHER_TO_RESTAURANT_DURATION` is NaN while the timestamps exist. That column isn't used in phase 1 metrics | 8 rows | report only |
| 11 | `IS_FIRST_DELIVERY = 1` on 7 rows, yet those Dashers have 142–1,403 lifetime deliveries. The flag looks unreliable; checked in phase 4 | 7 rows | report only |
| 12 | Lauran (Doorlanta) appears with 2 vehicles. Vehicle is treated per delivery | 1 Dasher | CLAUDE.md |
| 13 | Lifetime columns are constant within every Dasher (verified) | — | supports `.last()` later |
| 14 | Columns excluded by design: `COMPOSITE_SCORE`, `NUM_ON_TIME_DELIVERIES` (40.9% missing), `DASHER_TO_STORE_TIME_CATEGORY` (26.8% missing). `CANCELLED_AT` is 100% empty. All are kept in the export, used in no metric | 4 cols | CLAUDE.md |
| 15 | `DELIVERY_RATING` missing on 616 rows (51.3%): those deliveries were not rated. Left as NaN | 616 rows | rated-only denominator later |
| 16 | `assign_lag_min`, `buffer_min` and `late_vs_estimate` are computed on all rows. Filtering to ASAP happens at the point of use | — | CLAUDE.md metrics 6–8 |
| 17 | All dates sit in 2055 (fictional dataset). The year is immaterial: every metric is a difference between timestamps or a count inside the 13-day window, so nothing depends on the absolute date | — | report only |
| 18 | Doorlanta's deliveries end on June 8: its 600 orders span 8 days (~75/day), while Dashattan's span June 1–13 (~46/day). June 9–13 is Dashattan-only. Dataset fact; any "June 1–13" statement needs this caveat (sweep item 18) | 600 rows | PROMOTE-APPENDIX |

### 1.9 Export
Timezones are stripped before writing, because Excel can't store tz-aware datetimes.

In [15]:
export = df.copy()
for col in TIMESTAMP_COLS:
    export[col] = export[col].dt.tz_localize(None)
export.to_excel("export_01_cleaned.xlsx", sheet_name="cleaned_deliveries", index=False)
print("wrote export_01_cleaned.xlsx:", export.shape)

wrote export_01_cleaned.xlsx: (1200, 50)


## 2. Verification

**What this section does:** (a) documents the evidence behind the four columns excluded by design, (b) sets up the
reconciliation registry that every statistic in §3–§4 is logged into, and (c) previews the Dasher gates so the
per-Dasher missing-wait table can be shown for gate-passers.

**What it feeds:** the consolidated reconciliation table vs `reference/KEY_NUMBERS.md` is rendered at the end of §4
(§4.8), once all statistics have been computed. The gate preview is visibility only; Phase 3 re-implements the gates
from the spec and asserts it gets the same answer.

In [16]:
recon = []

def register(section, metric, expected, computed, tol, note=""):
    """Log one KEY_NUMBERS statistic. expected=None means the figure is not derivable from data/."""
    if expected is None:
        match = "n/a"
    else:
        match = "yes" if abs(float(expected) - float(computed)) <= tol else "NO"
    recon.append({"section": section, "metric": metric, "expected": expected,
                  "computed": None if computed is None else round(float(computed), 2),
                  "tol": tol, "match": match, "note": note})

asap = df[df["ASAP"] == 1]
by_market = df.groupby("MARKET_NAME")
by_market_asap = asap.groupby("MARKET_NAME")
print("ASAP rows:", len(asap), "| scheduled rows:", (df["ASAP"] == 0).sum())

ASAP rows: 1126 | scheduled rows: 74


### 2.1 Evidence for the excluded columns
`NUM_ON_TIME_DELIVERIES`, `COMPOSITE_SCORE`, `DASHER_TO_STORE_TIME_CATEGORY` and `CANCELLED_AT` are excluded from
every metric. The numbers below are the reason.

In [17]:
implied_ontime = 100 * df["NUM_ON_TIME_DELIVERIES"] / df["NUM_DELIVERIES"]
ontime_tbl = pd.DataFrame({
    "rows missing": by_market["NUM_ON_TIME_DELIVERIES"].apply(lambda s: s.isna().sum()),
    "% missing": (100 * by_market["NUM_ON_TIME_DELIVERIES"].apply(lambda s: s.isna().mean())).round(1),
    "implied lifetime on-time % (min)": implied_ontime.groupby(df["MARKET_NAME"]).min().round(1),
    "implied lifetime on-time % (max)": implied_ontime.groupby(df["MARKET_NAME"]).max().round(1),
})
ontime_tbl.loc["Both"] = [df["NUM_ON_TIME_DELIVERIES"].isna().sum(),
                          round(100 * df["NUM_ON_TIME_DELIVERIES"].isna().mean(), 1),
                          round(implied_ontime.min(), 1), round(implied_ontime.max(), 1)]
show("NUM_ON_TIME_DELIVERIES: missingness and the lifetime on-time rate it implies", ontime_tbl)
register("Data quality", "NUM_ON_TIME_DELIVERIES % missing", 41, ontime_tbl.loc["Both", "% missing"], 0.5)
register("Data quality", "Implied lifetime on-time % (min)", 1, ontime_tbl.loc["Both", "implied lifetime on-time % (min)"], 0.5)
register("Data quality", "Implied lifetime on-time % (max)", 77, ontime_tbl.loc["Both", "implied lifetime on-time % (max)"], 0.5)

corr_tbl = pd.DataFrame({
    "Pearson r with COMPOSITE_SCORE": {
        "DELIVERY_RATING (per order)": df["COMPOSITE_SCORE"].corr(df["DELIVERY_RATING"]),
        "COMPOSITE_STAR_RATING (lifetime)": df["COMPOSITE_SCORE"].corr(df["COMPOSITE_STAR_RATING"]),
    }
}).round(3)
show("COMPOSITE_SCORE: correlation with the two documented rating columns", corr_tbl)
register("Data quality", "r(COMPOSITE_SCORE, DELIVERY_RATING)", 0.05, corr_tbl.iloc[0, 0], 0.01)
register("Data quality", "r(COMPOSITE_SCORE, COMPOSITE_STAR_RATING)", 0.09, corr_tbl.iloc[1, 0], 0.01)

other_tbl = pd.DataFrame({
    "% missing": {
        "CANCELLED_AT": 100 * df["CANCELLED_AT"].isna().mean(),
        "DASHER_TO_STORE_TIME_CATEGORY": 100 * df["DASHER_TO_STORE_TIME_CATEGORY"].isna().mean(),
    }
}).round(1)
show("Other excluded columns", other_tbl)
register("Data quality", "CANCELLED_AT % empty", 100, other_tbl.loc["CANCELLED_AT", "% missing"], 0)
register("Data quality", "Junk trailing rows removed", 6, len(junk_rows), 0)

**NUM_ON_TIME_DELIVERIES: missingness and the lifetime on-time rate it implies**

,rows missing,% missing,implied lifetime on-time % (min),implied lifetime on-time % (max)
MARKET_NAME,,,,
Dashattan,415,69.2,0.8,26.6
Doorlanta,76,12.7,2.3,77.2
Both,491,40.9,0.8,77.2


**COMPOSITE_SCORE: correlation with the two documented rating columns**

,Pearson r with COMPOSITE_SCORE
DELIVERY_RATING (per order),0.047
COMPOSITE_STAR_RATING (lifetime),0.089


**Other excluded columns**

,% missing
CANCELLED_AT,100.0
DASHER_TO_STORE_TIME_CATEGORY,26.8


### 2.2 Dasher gate preview
Gates from the spec: vehicle set contains car or motorcycle · lifetime deliveries ≥ 100 (Dashattan) / 300 (Doorlanta)
· ASAP sample ≥ 5 / 10 · any fraud flag → `fraud_review` tag (never excludes). Lifetime columns are constant per
Dasher (§1.2), so `.last()` is safe. This table is the input to §2.3 and to the gate-count reconciliation; the score
itself is Phase 3.

In [18]:
dasher_all = df.groupby(["MARKET_NAME", "DASHER"]).agg(
    n_orders=("DELIVERY_NAME", "size"),
    lifetime_n=("NUM_DELIVERIES", "last"),
    lifetime_star=("COMPOSITE_STAR_RATING", "last"),
    fraud_flags=("FRAUDULENT", "sum"),
    vehicles=("VEHICLE", lambda s: "/".join(sorted(set(s)))),
)
dasher_all["asap_n"] = asap.groupby(["MARKET_NAME", "DASHER"]).size().reindex(dasher_all.index).fillna(0).astype(int)
dasher_all = dasher_all.reset_index()

LIFETIME_MIN = {"Dashattan": 100, "Doorlanta": 300}
ASAP_MIN = {"Dashattan": 5, "Doorlanta": 10}
dasher_all["gate_vehicle"] = dasher_all["vehicles"].str.contains("car|motorcycle")
dasher_all["gate_lifetime"] = dasher_all["lifetime_n"] >= dasher_all["MARKET_NAME"].map(LIFETIME_MIN)
dasher_all["gate_sample"] = dasher_all["asap_n"] >= dasher_all["MARKET_NAME"].map(ASAP_MIN)
dasher_all["gate_pass"] = dasher_all["gate_vehicle"] & dasher_all["gate_lifetime"] & dasher_all["gate_sample"]
dasher_all["fraud_review"] = dasher_all["fraud_flags"] > 0

gate_counts = dasher_all.groupby("MARKET_NAME").agg(
    dashers=("DASHER", "size"),
    pass_vehicle=("gate_vehicle", "sum"),
    pass_lifetime=("gate_lifetime", "sum"),
    pass_sample=("gate_sample", "sum"),
    pass_all_gates=("gate_pass", "sum"),
    fraud_review=("fraud_review", "sum"),
)
show("Gate pass counts per market (city-calibrated thresholds 100/5 and 300/10)", gate_counts)
register("Q1", "City-calibrated gate passers, Dashattan", 5, gate_counts.loc["Dashattan", "pass_all_gates"], 0)
register("Q1", "City-calibrated gate passers, Doorlanta", 15, gate_counts.loc["Doorlanta", "pass_all_gates"], 0)

**Gate pass counts per market (city-calibrated thresholds 100/5 and 300/10)**

,dashers,pass_vehicle,pass_lifetime,pass_sample,pass_all_gates,fraud_review
MARKET_NAME,,,,,,
Dashattan,24,8,23,16,5,5
Doorlanta,30,30,18,16,15,3


In [19]:
# KEY_NUMBERS claims "uniform strict criteria qualify: 1 Dashattan / 8 Doorlanta". The full definition (CLAUDE.md,
# Phase 3) also requires severe-late rate <= 10% and adjusted on-time >= 85%, which are Phase 3 quantities.
# Here: every pairing of the two gate thresholds applied to BOTH cities, gates only.
uniform_rows = []
for life_thr in (100, 300):
    for asap_thr in (5, 10):
        passes = (dasher_all["gate_vehicle"]
                  & (dasher_all["lifetime_n"] >= life_thr)
                  & (dasher_all["asap_n"] >= asap_thr))
        counts = dasher_all.loc[passes].groupby("MARKET_NAME").size()
        uniform_rows.append({"lifetime >=": life_thr, "asap_n >=": asap_thr,
                             "Dashattan": counts.get("Dashattan", 0), "Doorlanta": counts.get("Doorlanta", 0)})
uniform_tbl = pd.DataFrame(uniform_rows)
show("Uniform gate thresholds applied to both cities, gates only (the full uniform-strict check is Phase 3)", uniform_tbl)
strict = uniform_tbl[(uniform_tbl["lifetime >="] == 300) & (uniform_tbl["asap_n >="] == 10)].iloc[0]
register("Q1", "Uniform strict qualifiers, Dashattan (expect 1)", None, None, 0,
         f"deferred to Phase 3: gates alone give {strict['Dashattan']}; definition adds severe-late <= 10% and adjusted on-time >= 85%")
register("Q1", "Uniform strict qualifiers, Doorlanta (expect 8)", None, None, 0,
         f"deferred to Phase 3: gates alone give {strict['Doorlanta']}; same definition")

**Uniform gate thresholds applied to both cities, gates only (the full uniform-strict check is Phase 3)**

,lifetime >=,asap_n >=,Dashattan,Doorlanta
0,100,5,5,22
1,100,10,3,16
2,300,5,5,17
3,300,10,3,15


### 2.3 Visibility: missing wait per gate-passing Dasher
A missing wait means zero forgiveness (cleaning log #6). This table shows how many of each gate-passer's ASAP orders
carry that rule, so it's clear whether it concentrates on anyone near a tier boundary. **Visibility only, no metric
changes.** The two columns on the right split the missing waits into the two causes from §1.4.

In [20]:
asap_wait = asap.assign(
    wait_missing=asap["wait_min"].isna(),
    arrival_not_logged=asap["DASHER_AT_STORE_TIME"].isna(),
)
asap_wait["arrival_after_pickup"] = asap_wait["wait_missing"] & ~asap_wait["arrival_not_logged"]
missing_wait_tbl = asap_wait.groupby(["MARKET_NAME", "DASHER"]).agg(
    asap_n=("wait_missing", "size"),
    missing_wait_n=("wait_missing", "sum"),
    no_arrival_logged=("arrival_not_logged", "sum"),
    arrival_after_pickup=("arrival_after_pickup", "sum"),
).reset_index()
missing_wait_tbl["missing_wait_share"] = (missing_wait_tbl["missing_wait_n"] / missing_wait_tbl["asap_n"]).round(3)
missing_wait_tbl = missing_wait_tbl.merge(dasher_all[["MARKET_NAME", "DASHER", "gate_pass"]])
missing_wait_gate = (missing_wait_tbl[missing_wait_tbl["gate_pass"]]
                     .drop(columns="gate_pass")
                     .sort_values(["MARKET_NAME", "missing_wait_share"], ascending=[True, False])
                     .reset_index(drop=True))
show("Gate-passing Dashers: ASAP orders with missing wait (= zero forgiveness)", missing_wait_gate)

**Gate-passing Dashers: ASAP orders with missing wait (= zero forgiveness)**

,MARKET_NAME,DASHER,asap_n,missing_wait_n,no_arrival_logged,arrival_after_pickup,missing_wait_share
0,Dashattan,KC,138,9,9,0,0.065
1,Dashattan,Samm,44,2,2,0,0.045
2,Dashattan,Bill,8,0,0,0,0.000
3,Dashattan,Bret,17,0,0,0,0.000
4,Dashattan,Carlee,6,0,0,0,0.000
5,Doorlanta,Matt,17,6,3,3,0.353
6,Doorlanta,Cate,70,24,23,1,0.343
7,Doorlanta,Lauran,15,5,5,0,0.333
8,Doorlanta,Katelynn,61,17,13,4,0.279
9,Doorlanta,Abhai,33,9,6,3,0.273


## 3. Market comparison

**What this section does:** recomputes the market-overview and Q1/Q2 market facts in `KEY_NUMBERS.md`: the per-city
summary, Dashattan's vehicle mix and KC's concentration, and the hourly demand profile.

**What it feeds:** the market overview slide, the Q1 vehicle/concentration callouts, the Q2 onboarding-window
claim, and the `city_summary` export sheet.

### 3.1 City summary
Raw late rate uses all orders; the adjusted late rate is ASAP-only, per the spec. Durations are converted from
seconds to minutes.

In [21]:
city_summary = pd.DataFrame({
    "dashers": by_market["DASHER"].nunique(),
    "deliveries": by_market.size(),
    "raw late % (all orders)": 100 * by_market["is_late"].mean(),
    "adjusted late % (ASAP)": 100 * by_market_asap["is_late_adj"].mean(),
    "median confirm-to-deliver min": by_market["CONFIRM_TO_DELIVER_DURATION"].median() / 60,
    "avg delivery rating": by_market["DELIVERY_RATING"].mean(),
    "median travel-to-customer min": by_market["RESTAURANT_TO_CUSTOMER_DURATION"].median() / 60,
    "p90 travel-to-customer min": by_market["RESTAURANT_TO_CUSTOMER_DURATION"].quantile(0.9) / 60,
}).round(2)
show("City summary (June 1–13)", city_summary.T)

expected_city = {
    "dashers": (24, 30, 0),
    "raw late % (all orders)": (19, 16, 0.5),
    "adjusted late % (ASAP)": (14, 12, 0.5),
    "median confirm-to-deliver min": (32, 35, 0.5),
    "avg delivery rating": (4.55, 4.56, 0.01),
    "median travel-to-customer min": (12.1, 13.8, 0.1),
    "p90 travel-to-customer min": (23, 23, 1.0),
}
for metric, (exp_dash, exp_door, tol) in expected_city.items():
    register("Market overview", f"{metric}, Dashattan", exp_dash, city_summary.loc["Dashattan", metric], tol)
    register("Market overview", f"{metric}, Doorlanta", exp_door, city_summary.loc["Doorlanta", metric], tol)

**City summary (June 1–13)**

MARKET_NAME,Dashattan,Doorlanta
dashers,24.00,30.00
deliveries,600.00,600.00
raw late % (all orders),19.33,15.67
adjusted late % (ASAP),13.77,12.17
median confirm-to-deliver min,32.46,35.42
avg delivery rating,4.55,4.56
median travel-to-customer min,12.06,13.78
p90 travel-to-customer min,23.12,23.68


### 3.2 Dashattan vehicle mix and concentration
Vehicle is per delivery. "Car Dashers" are Dashers whose vehicle set includes a car.

In [22]:
dash = df[df["MARKET_NAME"] == "Dashattan"]
vehicle_mix = pd.DataFrame({
    "deliveries": dash["VEHICLE"].value_counts(),
    "% of deliveries": (100 * dash["VEHICLE"].value_counts(normalize=True)).round(1),
})
show("Dashattan: deliveries by vehicle", vehicle_mix)

dash_vehicle_sets = dasher_all.loc[dasher_all["MARKET_NAME"] == "Dashattan", "vehicles"].value_counts().rename("dashers")
show("Dashattan: Dashers by vehicle set", dash_vehicle_sets.to_frame())

dash_car = dash[dash["VEHICLE"] == "car"]
kc_share = 100 * (dash_car["DASHER"] == "KC").mean()
kc = dash[dash["DASHER"] == "KC"]
kc_tbl = pd.DataFrame({"value": {
    "Dashattan car deliveries": len(dash_car),
    "of which KC": (dash_car["DASHER"] == "KC").sum(),
    "KC share of car deliveries %": round(kc_share, 1),
    "KC observed deliveries (all)": len(kc),
    "KC fraud flags": int(kc["FRAUDULENT"].sum()),
    "KC fraud flag rate %": round(100 * kc["FRAUDULENT"].mean(), 1),
}})
show("KC concentration and fraud flags", kc_tbl)

register("Q1", "Dashattan % bicycle deliveries", 62, vehicle_mix.loc["bicycle", "% of deliveries"], 0.5)
register("Q1", "Dashattan car Dashers", 8, dash_vehicle_sets.get("car", 0), 0)
register("Q1", "KC share of Dashattan car deliveries %", 62, kc_share, 0.5)
register("Q1", "KC fraud flags", 2, kc_tbl.loc["KC fraud flags", "value"], 0)
register("Q1", "KC observed deliveries", 143, kc_tbl.loc["KC observed deliveries (all)", "value"], 0)
register("Q1", "KC fraud flag rate %", 1.4, kc_tbl.loc["KC fraud flag rate %", "value"], 0.05)

**Dashattan: deliveries by vehicle**

,deliveries,% of deliveries
VEHICLE,,
bicycle,369,61.5
car,231,38.5


**Dashattan: Dashers by vehicle set**

,dashers
vehicles,
bicycle,16
car,8


**KC concentration and fraud flags**

,value
Dashattan car deliveries,231.0
of which KC,143.0
KC share of car deliveries %,61.9
KC observed deliveries (all),143.0
KC fraud flags,2.0
KC fraud flag rate %,1.4


### 3.3 Hourly demand profile (local time)
Deliveries by local hour of completion across both cities, used for the Q2 onboarding-window claim
(lunch 11–14, dinner 18–20).

In [23]:
hourly = df.groupby(["hour_local", "MARKET_NAME"]).size().unstack(fill_value=0)
hourly["both"] = hourly.sum(axis=1)
hourly["% of all"] = (100 * hourly["both"] / len(df)).round(1)
show("Deliveries by local hour", hourly.T)

window_tbl = pd.DataFrame({"deliveries": {
    "lunch 11–14 (11:00–14:59)": df["hour_local"].between(11, 14).sum(),
    "dinner 18–20 (18:00–20:59)": df["hour_local"].between(18, 20).sum(),
    "all other hours": (~df["hour_local"].between(11, 14) & ~df["hour_local"].between(18, 20)).sum(),
}})
window_tbl["% of all"] = (100 * window_tbl["deliveries"] / len(df)).round(1)
show("Share of deliveries in the two peak windows", window_tbl)
top3 = hourly["both"].sort_values(ascending=False).head(3).index.tolist()
print("Three busiest local hours:", top3)
register("Q2", "Peak windows = lunch 11–14 & dinner 18–20 (busiest hours fall inside them)", 1,
         int(all(h in list(range(11, 15)) + list(range(18, 21)) for h in top3)), 0,
         "window boundary differs: data says dinner runs 18–21 (busiest hours " + str(top3) + ")")

**Deliveries by local hour**

hour_local,0,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
MARKET_NAME,,,,,,,,,,,,,,,,,
Dashattan,1.0,1.0,12.0,26.0,43.0,65.0,49.0,40.0,22.0,23.0,26.0,47.0,72.0,66.0,65.0,32.0,10.0
Doorlanta,0.0,0.0,0.0,5.0,32.0,53.0,55.0,36.0,28.0,34.0,37.0,67.0,86.0,84.0,58.0,24.0,1.0
both,1.0,1.0,12.0,31.0,75.0,118.0,104.0,76.0,50.0,57.0,63.0,114.0,158.0,150.0,123.0,56.0,11.0
% of all,0.1,0.1,1.0,2.6,6.2,9.8,8.7,6.3,4.2,4.8,5.2,9.5,13.2,12.5,10.2,4.7,0.9


**Share of deliveries in the two peak windows**

,deliveries,% of all
lunch 11–14 (11:00–14:59),373,31.1
dinner 18–20 (18:00–20:59),422,35.2
all other hours,405,33.8


Three busiest local hours: [19, 20, 21]


### 3.4 Q2 figures not derivable from the data
The density figures (90,909 vs 3,731 residents/sq mi) are external. The satchel sizing chain (1,000 daily food orders →
350 widget orders → ÷6 per Dasher-day → ~58 Dashers → ×1.3 → ~76 satchels) is an assumption chain, not a computation
from `data/`. Both are logged as *n/a* so the reconciliation table is complete.

In [24]:
register("Q2", "Population density (external)", None, None, 0, "not in data/; external source")
register("Q2", "Satchel sizing chain 350 → 58 → 76 (assumptions)", None, None, 0, "assumption chain, not computed from data/")
print("logged 2 non-data figures as n/a")

logged 2 non-data figures as n/a


## 4. Measurement

**What this section does:** recomputes the measurement claims: why the quoted time is the fair benchmark, how much
lateness traces to merchant holds and slow dispatch, scheduled-vs-ASAP and batched-vs-unbatched late rates, the
wait-adjustment effect on named Dashers, and the Q3 baselines. It ends with the consolidated reconciliation table.

**What it feeds:** the 2a Measurement slide, the Q3 framework and launch rules, and the Phase 3 scoring inputs.

### 4.1 Quoted time vs raw estimate
`buffer_min` is the cushion the quote adds on top of the raw estimate. `late_vs_estimate` is what the late rate would be
if measured against the raw estimate instead.

In [25]:
promise_tbl = pd.DataFrame({
    "orders": df.groupby("ASAP").size(),
    "mean quote buffer min": df.groupby("ASAP")["buffer_min"].mean().round(2),
    "late vs raw estimate %": (100 * df.groupby("ASAP")["late_vs_estimate"].mean()).round(1),
    "late vs quoted %": (100 * df.groupby("ASAP")["is_late"].mean()).round(1),
    "adjusted late vs quoted %": (100 * df.groupby("ASAP")["is_late_adj"].mean()).round(1),
}).rename(index={0: "scheduled (ASAP=0)", 1: "ASAP (ASAP=1)"})
show("Promise logic by order type", promise_tbl)
register("Q3", "Quote buffer min, scheduled", 0, promise_tbl.loc["scheduled (ASAP=0)", "mean quote buffer min"], 0.1)
register("Q3", "Quote buffer min, ASAP", 17, promise_tbl.loc["ASAP (ASAP=1)", "mean quote buffer min"], 0.5)
register("Q3", "Late vs raw estimate %, ASAP", 56, promise_tbl.loc["ASAP (ASAP=1)", "late vs raw estimate %"], 0.5)
register("Q3", "Raw late %, scheduled", 27, promise_tbl.loc["scheduled (ASAP=0)", "late vs quoted %"], 0.5)
register("Q3", "Raw late %, ASAP", 17, promise_tbl.loc["ASAP (ASAP=1)", "late vs quoted %"], 0.5)

**Promise logic by order type**

,orders,mean quote buffer min,late vs raw estimate %,late vs quoted %,adjusted late vs quoted %
ASAP,,,,,
scheduled (ASAP=0),74,0.00,27.0,27.0,20.3
ASAP (ASAP=1),1126,17.38,55.6,16.9,13.0


### 4.2 Scheduled orders
Volume share, orders per Dasher, and the late rate they would have had with a synthetic 17-minute buffer (the ASAP
mean) added to the quote.

In [26]:
sched = df[df["ASAP"] == 0]
sched_tbl = pd.DataFrame({"value": {
    "scheduled orders": len(sched),
    "% of all orders": round(100 * len(sched) / len(df), 1),
    "median scheduled orders per Dasher (Dashers with any)": sched.groupby(["MARKET_NAME", "DASHER"]).size().median(),
    "raw late %": round(100 * sched["is_late"].mean(), 1),
    "late % with synthetic 17-min buffer": round(100 * ((sched["late_min"] - 17) > 0).mean(), 1),
}})
show("Scheduled orders", sched_tbl)
register("Q3", "Scheduled share of volume %", 6.2, sched_tbl.loc["% of all orders", "value"], 0.1)
register("Q3", "Median scheduled orders per Dasher", 2, sched_tbl.loc["median scheduled orders per Dasher (Dashers with any)", "value"], 0)
register("Q3", "Scheduled late % with synthetic 17-min buffer", 1, sched_tbl.loc["late % with synthetic 17-min buffer", "value"], 0.5,
         "KEY_NUMBERS says ~1%")

**Scheduled orders**

,value
scheduled orders,74.0
% of all orders,6.2
median scheduled orders per Dasher (Dashers with any),2.0
raw late %,27.0
late % with synthetic 17-min buffer,1.4


### 4.3 Lateness attribution: merchant holds
Share of late deliveries (raw, vs quoted) where the Dasher waited more than 10 minutes at the merchant.

In [27]:
late_orders = df[df["is_late"]]
hold_tbl = pd.DataFrame({"value": {
    "late deliveries (raw)": len(late_orders),
    "of which merchant hold > 10 min": (late_orders["wait_min"] > 10).sum(),
    "share with hold > 10 min %": round(100 * (late_orders["wait_min"] > 10).mean(), 1),
    "of which wait missing (counted as no hold)": late_orders["wait_min"].isna().sum(),
}})
show("Late deliveries and merchant holds", hold_tbl)
register("Q1", "Share of late deliveries with merchant hold > 10 min %", 40, hold_tbl.loc["share with hold > 10 min %", "value"], 0.5)

**Late deliveries and merchant holds**

,value
late deliveries (raw),210.0
of which merchant hold > 10 min,84.0
share with hold > 10 min %,40.0
of which wait missing (counted as no hold),43.0


**Long holds dent ratings even when the delivery lands on time.** Rated, on-time (adjusted) deliveries only, by merchant-wait
band; promoted here from sweep item 27 (§8), same definitions.

In [28]:
ontime_rated = df[~df["is_late_adj"]].copy()
ontime_rated["wait_band"] = (pd.cut(ontime_rated["wait_min"], [0, 5, 10, 20, 1e9], right=False, labels=["0–5", "5–10", "10–20", "20+"])
                             .astype(object).fillna("missing"))
rating_by_wait_ontime = ontime_rated.groupby("wait_band").agg(
    orders=("DELIVERY_RATING", "size"), rated=("DELIVERY_RATING", "count"), mean_rating=("DELIVERY_RATING", "mean"),
    low_rating_pct=("DELIVERY_RATING", lambda s: 100 * (s <= 3).sum() / s.notna().sum()),
).round(2).loc[["0–5", "5–10", "10–20", "20+", "missing"]]
show("On-time deliveries: rating by merchant-wait band (minutes)", rating_by_wait_ontime)

**On-time deliveries: rating by merchant-wait band (minutes)**

,orders,rated,mean_rating,low_rating_pct
wait_band,,,,
0–5,317,154,4.71,7.79
5–10,248,119,4.53,11.76
10–20,208,104,4.56,10.58
20+,118,65,4.49,13.85
missing,148,74,4.73,9.46


### 4.4 Lateness attribution: dispatch speed
ASAP orders split into quartiles of assignment lag (order created → Dasher assigned). Raw late rate by quartile.

In [29]:
lag = asap[["assign_lag_min", "is_late", "is_late_adj"]].copy()
lag["lag quartile"] = pd.qcut(lag["assign_lag_min"], 4, labels=["Q1 fastest", "Q2", "Q3", "Q4 slowest"])
lag_tbl = lag.groupby("lag quartile", observed=True).agg(
    orders=("is_late", "size"),
    lag_min_low=("assign_lag_min", "min"),
    lag_min_high=("assign_lag_min", "max"),
    raw_late_pct=("is_late", lambda s: 100 * s.mean()),
    adjusted_late_pct=("is_late_adj", lambda s: 100 * s.mean()),
).round(1)
show("ASAP late rate by assignment-lag quartile", lag_tbl)
register("Q1", "Raw late %, fastest lag quartile", 12, lag_tbl.loc["Q1 fastest", "raw_late_pct"], 0.5)
register("Q1", "Raw late %, slowest lag quartile", 26, lag_tbl.loc["Q4 slowest", "raw_late_pct"], 0.5)

**ASAP late rate by assignment-lag quartile**

,orders,lag_min_low,lag_min_high,raw_late_pct,adjusted_late_pct
lag quartile,,,,,
Q1 fastest,282,0.0,0.3,12.1,7.4
Q2,281,0.3,4.1,14.6,11.4
Q3,281,4.1,8.2,14.9,8.9
Q4 slowest,282,8.4,131.0,25.9,24.1


### 4.5 Batching
Raw late rate for batched vs unbatched orders per city, on all orders and on ASAP only (both versions appear in
KEY_NUMBERS).

In [30]:
def batch_table(frame):
    t = frame.groupby(["MARKET_NAME", "WAS_BATCHED"]).agg(
        orders=("is_late", "size"),
        raw_late_pct=("is_late", lambda s: 100 * s.mean()),
        adjusted_late_pct=("is_late_adj", lambda s: 100 * s.mean()),
    ).round(1)
    t.index = t.index.set_levels(["unbatched", "batched"], level=1)
    return t

batch_all = batch_table(df)
batch_asap = batch_table(asap)
show("Batched vs unbatched, all orders", batch_all)
show("Batched vs unbatched, ASAP only", batch_asap)
for market, exp_b, exp_u in [("Dashattan", 35, 16), ("Doorlanta", 19, 15)]:
    register("Q3", f"Raw late % batched (all orders), {market}", exp_b, batch_all.loc[(market, "batched"), "raw_late_pct"], 0.5)
    register("Q3", f"Raw late % unbatched (all orders), {market}", exp_u, batch_all.loc[(market, "unbatched"), "raw_late_pct"], 0.5)
for market, exp_b, exp_u in [("Dashattan", 33, 16), ("Doorlanta", 16, 15)]:
    register("Q3", f"Raw late % batched (ASAP), {market}", exp_b, batch_asap.loc[(market, "batched"), "raw_late_pct"], 0.5)
    register("Q3", f"Raw late % unbatched (ASAP), {market}", exp_u, batch_asap.loc[(market, "unbatched"), "raw_late_pct"], 0.5)

**Batched vs unbatched, all orders**

orders  raw_late_pct  adjusted_late_pct
MARKET_NAME WAS_BATCHED                                         
Dashattan   unbatched       496          16.1               11.9
            batched         104          34.6               26.0
Doorlanta   unbatched       547          15.4               12.2
            batched          53          18.9               15.1

**Batched vs unbatched, ASAP only**

orders  raw_late_pct  adjusted_late_pct
MARKET_NAME WAS_BATCHED                                         
Dashattan   unbatched       463          15.8               11.7
            batched          96          33.3               24.0
Doorlanta   unbatched       517          14.9               12.0
            batched          50          16.0               14.0

### 4.6 Wait-adjustment effect on named Dashers
Raw vs adjusted on-time rate, ASAP only, for the four Dashers named in KEY_NUMBERS. Rank changes depend on the full
score and are checked in Phase 3.

In [31]:
ontime_by_dasher = asap.groupby(["MARKET_NAME", "DASHER"]).agg(
    asap_n=("is_late", "size"),
    ontime_raw_pct=("is_late", lambda s: 100 * (1 - s.mean())),
    ontime_adj_pct=("is_late_adj", lambda s: 100 * (1 - s.mean())),
).round(1)
ontime_by_dasher["change_pts"] = (ontime_by_dasher["ontime_adj_pct"] - ontime_by_dasher["ontime_raw_pct"]).round(1)
named = ontime_by_dasher.loc[[("Doorlanta", "Josh"), ("Doorlanta", "Merissa"), ("Doorlanta", "Lauran"), ("Doorlanta", "Matt")]]
show("Named Dashers: on-time rate before and after the wait adjustment (ASAP only)", named)
register("Q1", "Josh on-time raw %", 80, named.loc[("Doorlanta", "Josh"), "ontime_raw_pct"], 0.5)
register("Q1", "Josh on-time adjusted %", 95, named.loc[("Doorlanta", "Josh"), "ontime_adj_pct"], 0.5)
register("Q1", "Merissa on-time change (pts)", 21, named.loc[("Doorlanta", "Merissa"), "change_pts"], 0.5)
register("Q1", "Lauran on-time change (pts)", 0, named.loc[("Doorlanta", "Lauran"), "change_pts"], 0.5, "'barely move'")
register("Q1", "Matt on-time change (pts)", 0, named.loc[("Doorlanta", "Matt"), "change_pts"], 0.5, "'barely move'")
register("Q1", "Josh rank 7 → 2", None, None, 0, "needs full score; checked in Phase 3")

**Named Dashers: on-time rate before and after the wait adjustment (ASAP only)**

asap_n  ontime_raw_pct  ontime_adj_pct  change_pts
MARKET_NAME DASHER                                                     
Doorlanta   Josh         20            80.0            95.0        15.0
            Merissa      19            57.9            78.9        21.0
            Lauran       15            73.3            73.3         0.0
            Matt         17            58.8            58.8         0.0

### 4.7 Q3 baselines
Adjusted late rate (ASAP only), long-hold share (all deliveries with wait > 10 min; missing wait counts as not long),
and low-rating share (rating ≤ 3 over rated deliveries only).

In [32]:
baselines = pd.DataFrame({
    "adjusted late % (ASAP)": 100 * by_market_asap["is_late_adj"].mean(),
    "long merchant hold % (wait > 10, all orders)": 100 * by_market["wait_min"].apply(lambda s: (s > 10).mean()),
    "low rating % (<= 3, of rated)": 100 * by_market["DELIVERY_RATING"].apply(lambda s: (s <= 3).sum() / s.notna().sum()),
    "rated deliveries": by_market["DELIVERY_RATING"].count(),
}).round(1)
show("Q3 baselines per market", baselines.T)
register("Q3", "Long hold % Dashattan", 30, baselines.loc["Dashattan", "long merchant hold % (wait > 10, all orders)"], 0.5)
register("Q3", "Long hold % Doorlanta", 30, baselines.loc["Doorlanta", "long merchant hold % (wait > 10, all orders)"], 0.5)
register("Q3", "Low rating % Dashattan", 11, baselines.loc["Dashattan", "low rating % (<= 3, of rated)"], 0.5)
register("Q3", "Low rating % Doorlanta", 13, baselines.loc["Doorlanta", "low rating % (<= 3, of rated)"], 0.5)

**Q3 baselines per market**

MARKET_NAME,Dashattan,Doorlanta
adjusted late % (ASAP),13.8,12.2
"long merchant hold % (wait > 10, all orders)",30.2,29.7
"low rating % (<= 3, of rated)",10.9,12.6
rated deliveries,267.0,317.0


### 4.8 Reconciliation table vs KEY_NUMBERS
Every statistic in `reference/KEY_NUMBERS.md` that this phase can compute, with the expected value, the computed value,
the tolerance used, and whether they match. *n/a* rows are figures not derivable from `data/` (external numbers,
assumptions) or deferred to a later phase. Rows marked **NO** are unreconciled and reported, not resolved.

The two *uniform strict* rows are *n/a* here: their definition includes severe-late rate and adjusted on-time, which are Phase 3 quantities, so they are verified in Phase 3 (§5) against the 1 / 8 expectation. The dinner-window row stays flagged: the stated 18–20 window ends an hour before the data's dinner peak does.

In [33]:
recon_tbl = pd.DataFrame(recon)
show("Reconciliation: KEY_NUMBERS vs computed", recon_tbl)
summary = recon_tbl["match"].value_counts()
print("Matched:", summary.get("yes", 0), "| Unreconciled:", summary.get("NO", 0), "| n/a:", summary.get("n/a", 0))
show("Unreconciled rows", recon_tbl[recon_tbl["match"] == "NO"])

**Reconciliation: KEY_NUMBERS vs computed**

,section,metric,expected,computed,tol,match,note
0,Data quality,NUM_ON_TIME_DELIVERIES % missing,41.00,40.90,0.50,yes,
1,Data quality,Implied lifetime on-time % (min),1.00,0.80,0.50,yes,
2,Data quality,Implied lifetime on-time % (max),77.00,77.20,0.50,yes,
3,Data quality,"r(COMPOSITE_SCORE, DELIVERY_RATING)",0.05,0.05,0.01,yes,
4,Data quality,"r(COMPOSITE_SCORE, COMPOSITE_STAR_RATING)",0.09,0.09,0.01,yes,
5,Data quality,CANCELLED_AT % empty,100.00,100.00,0.00,yes,
6,Data quality,Junk trailing rows removed,6.00,6.00,0.00,yes,
7,Q1,"City-calibrated gate passers, Dashattan",5.00,5.00,0.00,yes,
8,Q1,"City-calibrated gate passers, Doorlanta",15.00,15.00,0.00,yes,
9,Q1,"Uniform strict qualifiers, Dashattan (expect 1)",NaN,NaN,0.00,n/a,deferred to Phase 3: gates alone give 3; definition adds severe-late <= 10% and adjusted on-time >= 85%


Matched: 57 | Unreconciled: 1 | n/a: 5


**Unreconciled rows**

,section,metric,expected,computed,tol,match,note
31,Q2,Peak windows = lunch 11–14 & dinner 18–20 (busiest hours fall inside them),1.0,0.0,0.0,NO,"window boundary differs: data says dinner runs 18–21 (busiest hours [19, 20, 21])"


### 4.9 Chart data for the deck
**What this does:** assembles one small tidy table per deck visual, reusing the tables above with the same definitions
(no new metrics). The only new arithmetic is the vehicle mix for both cities and the late-delivery split by merchant hold.

**What it feeds:** one `viz_*` sheet per visual in `export_02_reconciled.xlsx`, each with a title row so it reads on its
own. `viz_sizing_funnel` is an assumption chain, not a computation from `data/`, and is labelled as such. After Phase 3,
`viz_eligibility` and `viz_scatter` are added (they need score, shrunk rating and the uniform-strict check).

In [34]:
# viz_lateness_attribution · late deliveries (raw, vs quoted) split by merchant hold; dispatch-lag block reuses lag_tbl (4.4)
late_split = pd.DataFrame({"late deliveries": {
    "merchant hold > 10 min": int((late_orders["wait_min"] > 10).sum()),
    "no long hold (wait <= 10 min or wait missing)": int((~(late_orders["wait_min"] > 10)).sum()),
}})
late_split["share %"] = (100 * late_split["late deliveries"] / late_split["late deliveries"].sum()).round(1)
late_split.loc["all late deliveries"] = [late_split["late deliveries"].sum(), 100.0]
show("viz_lateness_attribution · Late deliveries by merchant hold", late_split)
show("viz_lateness_attribution · Dispatch-lag quartiles (ASAP), reused from 4.4", lag_tbl)

**viz_lateness_attribution · Late deliveries by merchant hold**

,late deliveries,share %
merchant hold > 10 min,84,40.0
no long hold (wait <= 10 min or wait missing),126,60.0
all late deliveries,210,100.0


**viz_lateness_attribution · Dispatch-lag quartiles (ASAP), reused from 4.4**

,orders,lag_min_low,lag_min_high,raw_late_pct,adjusted_late_pct
lag quartile,,,,,
Q1 fastest,282,0.0,0.3,12.1,7.4
Q2,281,0.3,4.1,14.6,11.4
Q3,281,4.1,8.2,14.9,8.9
Q4 slowest,282,8.4,131.0,25.9,24.1


In [35]:
# viz_vehicle_mix · deliveries by vehicle by city (vehicle is per delivery; Lauran counts under both of her vehicles)
vehicle_mix_city = df.groupby(["MARKET_NAME", "VEHICLE"]).size().rename("deliveries").reset_index()
vehicle_mix_city["% of city deliveries"] = (
    100 * vehicle_mix_city["deliveries"] / vehicle_mix_city.groupby("MARKET_NAME")["deliveries"].transform("sum")
).round(1)
dashers_using = (df.drop_duplicates(["MARKET_NAME", "DASHER", "VEHICLE"])
                   .groupby(["MARKET_NAME", "VEHICLE"]).size().rename("Dashers using vehicle").reset_index())
vehicle_mix_city = vehicle_mix_city.merge(dashers_using)
show("viz_vehicle_mix · Deliveries by vehicle by city", vehicle_mix_city)
show("viz_vehicle_mix · KC concentration block, reused from 3.2", kc_tbl)

**viz_vehicle_mix · Deliveries by vehicle by city**

,MARKET_NAME,VEHICLE,deliveries,% of city deliveries,Dashers using vehicle
0,Dashattan,bicycle,369,61.5,16
1,Dashattan,car,231,38.5,8
2,Doorlanta,car,597,99.5,30
3,Doorlanta,motorcycle,3,0.5,1


**viz_vehicle_mix · KC concentration block, reused from 3.2**

,value
Dashattan car deliveries,231.0
of which KC,143.0
KC share of car deliveries %,61.9
KC observed deliveries (all),143.0
KC fraud flags,2.0
KC fraud flag rate %,1.4


In [36]:
# viz_wait_adjustment · gate-passing Dashers only (the Josh / Merissa exhibit); ontime_by_dasher from 4.6
wait_adjustment = (ontime_by_dasher.reset_index()
                   .merge(dasher_all[["MARKET_NAME", "DASHER", "gate_pass"]])
                   .query("gate_pass").drop(columns="gate_pass")
                   .sort_values(["MARKET_NAME", "change_pts"], ascending=[True, False]).reset_index(drop=True))
show("viz_wait_adjustment · Raw vs adjusted on-time per gate-passing Dasher (ASAP only)", wait_adjustment)

**viz_wait_adjustment · Raw vs adjusted on-time per gate-passing Dasher (ASAP only)**

,MARKET_NAME,DASHER,asap_n,ontime_raw_pct,ontime_adj_pct,change_pts
0,Dashattan,Bret,17,35.3,52.9,17.6
1,Dashattan,Bill,8,62.5,75.0,12.5
2,Dashattan,Samm,44,79.5,84.1,4.6
3,Dashattan,KC,138,89.1,92.8,3.7
4,Dashattan,Carlee,6,66.7,66.7,0.0
5,Doorlanta,Merissa,19,57.9,78.9,21.0
6,Doorlanta,Josh,20,80.0,95.0,15.0
7,Doorlanta,Ash,12,66.7,75.0,8.3
8,Doorlanta,Hersh,25,72.0,80.0,8.0
9,Doorlanta,Joe,33,75.8,78.8,3.0


In [37]:
# viz_sizing_funnel · ASSUMPTION CHAIN, not computed from data/. Each step's assumption is stated beside its value.
FOOD_ORDERS_BASIS = 1000      # scaling basis: per 1,000 daily food orders
WIDGET_SHARE = 0.35           # widget demand = 35% of food demand (case prompt)
DELIVERIES_PER_DASHER_DAY = 6 # widget deliveries one Dasher completes per day
SATCHEL_BUFFER = 1.30         # 30% extra satchels for spares, repairs, churn
widget_orders = FOOD_ORDERS_BASIS * WIDGET_SHARE
dashers_needed = widget_orders / DELIVERIES_PER_DASHER_DAY
satchels = dashers_needed * SATCHEL_BUFFER
sizing_funnel = pd.DataFrame([
    ("daily food orders (basis)", FOOD_ORDERS_BASIS, FOOD_ORDERS_BASIS, "scaling basis per 1,000 daily food orders; not a forecast"),
    ("widget orders per day", widget_orders, round(widget_orders), f"assumes widget demand = {WIDGET_SHARE:.0%} of food demand"),
    ("Dashers needed per day", dashers_needed, round(dashers_needed), f"assumes {DELIVERIES_PER_DASHER_DAY} widget deliveries per Dasher-day"),
    ("satchels to procure", satchels, round(satchels), f"assumes {SATCHEL_BUFFER - 1:.0%} buffer for spares, repairs and churn"),
], columns=["step", "value", "rounded", "assumption"])
show("viz_sizing_funnel · Satchel sizing chain (ASSUMPTIONS, NOT COMPUTED FROM DATA)", sizing_funnel)

**viz_sizing_funnel · Satchel sizing chain (ASSUMPTIONS, NOT COMPUTED FROM DATA)**

,step,value,rounded,assumption
0,daily food orders (basis),1000.000000,1000,"scaling basis per 1,000 daily food orders; not a forecast"
1,widget orders per day,350.000000,350,assumes widget demand = 35% of food demand
2,Dashers needed per day,58.333333,58,assumes 6 widget deliveries per Dasher-day
3,satchels to procure,75.833333,76,"assumes 30% buffer for spares, repairs and churn"


### 4.10 Export
`cleaned_deliveries`, `city_summary` and `reconciliation` as before, then one `viz_*` sheet per deck visual. Each viz sheet starts with a title row; two-block sheets separate the blocks with a blank row.

In [38]:
from openpyxl.styles import Font

plain_sheets = {
    "cleaned_deliveries": (export, False),
    "city_summary": (city_summary, True),
    "reconciliation": (recon_tbl, False),
}
# viz sheets: list of (title, table, keep_index) blocks, written top to bottom
viz_sheets = {
    "viz_lateness_attribution": [
        ("Late deliveries (raw, vs quoted) split by merchant hold > 10 min", late_split, True),
        ("Footnote: raw late rate by assignment-lag quartile, ASAP orders", lag_tbl, True)],
    "viz_promise_gap": [
        ("Promise logic by order type: late vs quote, mean buffer (quoted - estimated), late vs raw estimate", promise_tbl, True)],
    "viz_vehicle_mix": [
        ("Deliveries by vehicle by city (counts and % of city deliveries)", vehicle_mix_city, False),
        ("KC concentration: Dashattan car deliveries, KC's share, fraud flags", kc_tbl, True)],
    "viz_hourly_demand": [
        ("Deliveries by local hour (Galaxy A, UTC-4) by city", hourly[["Dashattan", "Doorlanta", "both", "% of all"]], True)],
    "viz_batching": [
        ("ASAP-only late rate by batched flag by city", batch_asap, True)],
    "viz_wait_adjustment": [
        ("Gate-passing Dashers: raw vs adjusted on-time (ASAP only) and the change in points", wait_adjustment, False)],
    "viz_q3_baselines": [
        ("Q3 baselines per city: adjusted late (ASAP), long merchant hold share, low-rating share", baselines, True)],
    "viz_sizing_funnel": [
        ("ASSUMPTIONS, NOT COMPUTED FROM DATA: satchel sizing chain per 1,000 daily food orders", sizing_funnel, False)],
}

def write_workbook(path, plain, viz):
    """plain: {sheet: (table, keep_index)} written as-is. viz: {sheet: [(title, table, keep_index), ...]} written
    with a bold title row above each block and a blank row between blocks. Returns data rows per sheet."""
    with pd.ExcelWriter(path, engine="openpyxl") as xw:  # openpyxl: the bold title row needs ws.cell()
        for name, (table, keep_index) in plain.items():
            table.to_excel(xw, sheet_name=name, index=keep_index)
        for name, blocks in viz.items():
            row = 0
            for title, table, keep_index in blocks:
                pd.DataFrame([[title]]).to_excel(xw, sheet_name=name, startrow=row, index=False, header=False)
                xw.sheets[name].cell(row=row + 1, column=1).font = Font(bold=True)
                table.to_excel(xw, sheet_name=name, startrow=row + 1, index=keep_index)
                row += len(table) + 4   # title + header + rows + blank separator
    written = {name: len(t) for name, (t, _) in plain.items()}
    written.update({name: sum(len(t) for _, t, _ in blocks) for name, blocks in viz.items()})
    return pd.Series(written, name="data rows").to_frame()

show("wrote export_02_reconciled.xlsx", write_workbook("export_02_reconciled.xlsx", plain_sheets, viz_sheets))

**wrote export_02_reconciled.xlsx**

,data rows
cleaned_deliveries,1200
city_summary,2
reconciliation,63
viz_lateness_attribution,7
viz_promise_gap,2
viz_vehicle_mix,10
viz_hourly_demand,17
viz_batching,4
viz_wait_adjustment,20
viz_q3_baselines,2


## 5. Dasher scoring

**What this section does:** implements the Dasher-level spec from `CLAUDE.md` (ASAP-only performance, lifetime
profile, shrunk rating, four score components, gates, tiers), diffs every field against `reference/dasher_scorecard.csv`,
locks the result with an assert cell, traces KC, Josh and Bret end-to-end, shows the conditional bike cohort, and
reconciles the uniform-strict count against 1 / 8.

**What it feeds:** the Q1 selection slides, Appendix C scorecard, the Phase 5 sensitivity baseline, and
`export_03_scorecard.xlsx`. Column names follow the reference file so the diff reads one-to-one.

### 5.1 Dasher-level metrics
Performance is computed on ASAP orders only, then joined to the lifetime profile computed over all orders. The shrink
pulls each Dasher's lifetime star rating toward the market mean with k = 200 pseudo-deliveries; the market mean is the
mean of lifetime star over the deduplicated Dasher table (one row per Dasher, 24 + 30).

In [39]:
K_SHRINK = 200
GATE_LIFETIME_MIN = {"Dashattan": 100, "Doorlanta": 300}
GATE_ASAP_MIN = {"Dashattan": 5, "Doorlanta": 10}
TIER1_SIZE = {"Dashattan": 4, "Doorlanta": 10}
SEVERE_CAP = 0.30

# performance: filter ASAP == 1 FIRST, then group
perf = asap.groupby(["MARKET_NAME", "DASHER"]).agg(
    asap_n=("is_late_adj", "size"),
    ontime_raw=("is_late", lambda s: 1 - s.mean()),
    ontime_adj=("is_late_adj", lambda s: 1 - s.mean()),
    severe_late_rate=("severe", "mean"),
)
# lifetime profile over ALL orders; lifetime columns are constant per Dasher so .last() is exact
profile = df.groupby(["MARKET_NAME", "DASHER"]).agg(
    lifetime_deliveries=("NUM_DELIVERIES", "last"),
    lifetime_star=("COMPOSITE_STAR_RATING", "last"),
    sample_deliveries=("DELIVERY_NAME", "size"),
    fraud_flags=("FRAUDULENT", "sum"),
    vehicles=("VEHICLE", lambda s: "/".join(sorted(set(s)))),
    arrival_logged=("arrival_logged", "mean"),
)
scorecard = profile.join(perf, how="left")
market_mean_star = scorecard.groupby(level="MARKET_NAME")["lifetime_star"].mean()
scorecard["market_mean_star"] = scorecard.index.get_level_values("MARKET_NAME").map(market_mean_star)
n = scorecard["lifetime_deliveries"]
scorecard["shrunk_rating"] = (n * scorecard["lifetime_star"] + K_SHRINK * scorecard["market_mean_star"]) / (n + K_SHRINK)

show("Market mean of lifetime star (deduplicated Dasher table)", market_mean_star.round(4).to_frame())
print("Dashers:", len(scorecard), "| Dashers with no ASAP orders:", int(scorecard["asap_n"].isna().sum()))

**Market mean of lifetime star (deduplicated Dasher table)**

,lifetime_star
MARKET_NAME,
Dashattan,4.4717
Doorlanta,4.5876


Dashers: 54 | Dashers with no ASAP orders: 0


### 5.2 Score, gates, tiers
Score = 50·ontime_adj + 30·shrunk_rating/5 + 10·max(0, 1 − severe_rate/0.30) + 10·arrival_share.
Gates: widget-capable vehicle (car or motorcycle in the vehicle set) · lifetime ≥ 100 / 300 · asap_n ≥ 5 / 10.
Fraud flags only tag `fraud_review`. Tier 1 = top 4 / top 10 gate-passers by score; the rest are Tier 2.

In [40]:
scorecard["widget_capable"] = scorecard["vehicles"].str.contains("car|motorcycle")
scorecard["pts_reliability"] = 50 * scorecard["ontime_adj"]
scorecard["pts_quality"] = 30 * scorecard["shrunk_rating"] / 5
scorecard["pts_consistency"] = 10 * (1 - scorecard["severe_late_rate"] / SEVERE_CAP).clip(lower=0)
scorecard["pts_professionalism"] = 10 * scorecard["arrival_logged"]
scorecard["selection_score"] = (scorecard["pts_reliability"] + scorecard["pts_quality"]
                                + scorecard["pts_consistency"] + scorecard["pts_professionalism"])

markets = scorecard.index.get_level_values("MARKET_NAME")
scorecard["passes_gates"] = (scorecard["widget_capable"]
                             & (scorecard["lifetime_deliveries"] >= markets.map(GATE_LIFETIME_MIN))
                             & (scorecard["asap_n"].fillna(0) >= markets.map(GATE_ASAP_MIN)))
scorecard["fraud_review"] = scorecard["fraud_flags"] > 0

scorecard["tier"] = None
for market, passers in scorecard[scorecard["passes_gates"]].groupby(level="MARKET_NAME"):
    ranked = passers["selection_score"].sort_values(ascending=False)
    scorecard.loc[ranked.index, "tier"] = "Tier 2"
    scorecard.loc[ranked.index[:TIER1_SIZE[market]], "tier"] = "Tier 1"

scorecard = scorecard.reset_index()
show("Gate and tier counts per market", scorecard.groupby("MARKET_NAME").agg(
    dashers=("DASHER", "size"), widget_capable=("widget_capable", "sum"), passes_gates=("passes_gates", "sum"),
    tier_1=("tier", lambda s: (s == "Tier 1").sum()), tier_2=("tier", lambda s: (s == "Tier 2").sum()),
    fraud_review=("fraud_review", "sum")))
SCORE_COLS = ["pts_reliability", "pts_quality", "pts_consistency", "pts_professionalism", "selection_score"]
show("Scorecard: gate-passers ranked by score", scorecard[scorecard["passes_gates"]]
     .sort_values(["MARKET_NAME", "selection_score"], ascending=[True, False])
     [["MARKET_NAME", "DASHER", "asap_n", "ontime_adj", "shrunk_rating", "severe_late_rate", "arrival_logged"]
      + SCORE_COLS + ["tier", "fraud_review"]].round(3).reset_index(drop=True))

**Gate and tier counts per market**

,dashers,widget_capable,passes_gates,tier_1,tier_2,fraud_review
MARKET_NAME,,,,,,
Dashattan,24,8,5,4,1,5
Doorlanta,30,30,15,10,5,3


**Scorecard: gate-passers ranked by score**

,MARKET_NAME,DASHER,asap_n,ontime_adj,shrunk_rating,severe_late_rate,arrival_logged,pts_reliability,pts_quality,pts_consistency,pts_professionalism,selection_score,tier,fraud_review
0,Dashattan,KC,138,0.928,4.538,0.014,0.937,46.377,27.228,9.517,9.371,92.492,Tier 1,True
1,Dashattan,Samm,44,0.841,4.076,0.045,0.958,42.045,24.456,8.485,9.583,84.570,Tier 1,True
2,Dashattan,Carlee,6,0.667,4.511,0.000,1.000,33.333,27.067,10.000,10.000,80.400,Tier 1,False
3,Dashattan,Bill,8,0.750,4.464,0.125,1.000,37.500,26.782,5.833,10.000,80.115,Tier 1,False
4,Dashattan,Bret,17,0.529,3.954,0.294,1.000,26.471,23.724,0.196,10.000,60.390,Tier 2,True
5,Doorlanta,Ajay,11,1.000,4.574,0.000,1.000,50.000,27.443,10.000,10.000,97.443,Tier 1,False
6,Doorlanta,Josh,20,0.950,4.744,0.000,0.952,47.500,28.466,10.000,9.524,95.490,Tier 1,False
7,Doorlanta,Jagar,46,0.957,4.879,0.022,0.880,47.826,29.272,9.275,8.800,95.173,Tier 1,False
8,Doorlanta,Elon,76,0.947,4.612,0.000,0.878,47.368,27.672,10.000,8.780,93.820,Tier 1,False
9,Doorlanta,Abhai,33,0.909,4.565,0.000,0.824,45.455,27.389,10.000,8.235,91.079,Tier 1,False


### 5.3 Diff against the reference scorecard
Every shared column is compared Dasher by Dasher. The reference stores rates, components and the lifetime star rating
to 3 decimals (the raw data carries more), so those use a 0.001 tolerance; the score uses the spec's ±0.1; booleans, tier and vehicle strings must match exactly.

In [41]:
reference = pd.read_csv("reference/dasher_scorecard.csv")
KEYS = ["MARKET_NAME", "DASHER"]
NUMERIC_TOL = {
    "lifetime_deliveries": 0, "lifetime_star": 0.001, "sample_deliveries": 0,  # lifetime_star: reference rounds to 3 dp "fraud_flags": 0, "asap_n": 0,
    "arrival_logged": 0.001, "ontime_raw": 0.001, "ontime_adj": 0.001, "severe_late_rate": 0.001, "shrunk_rating": 0.001,
    "pts_reliability": 0.001, "pts_quality": 0.001, "pts_consistency": 0.001, "pts_professionalism": 0.001,
    "selection_score": 0.1,
}
EXACT = ["vehicles", "widget_capable", "passes_gates", "fraud_review", "tier"]

merged = scorecard.merge(reference, on=KEYS, how="outer", suffixes=("", "_ref"), indicator=True)
assert (merged["_merge"] == "both").all(), merged.loc[merged["_merge"] != "both", KEYS + ["_merge"]]

diff_rows, mismatches = [], []
for col, tol in NUMERIC_TOL.items():
    gap = (merged[col] - merged[f"{col}_ref"]).abs()
    bad = gap > tol
    diff_rows.append({"field": col, "tolerance": tol, "max_abs_diff": round(gap.max(), 5), "mismatches": int(bad.sum())})
    mismatches += [dict(zip(KEYS, r)) | {"field": col, "computed": c, "reference": e}
                   for r, c, e in zip(merged.loc[bad, KEYS].values, merged.loc[bad, col], merged.loc[bad, f"{col}_ref"])]
for col in EXACT:
    mine, theirs = merged[col].fillna("").astype(str), merged[f"{col}_ref"].fillna("").astype(str)
    bad = mine != theirs
    diff_rows.append({"field": col, "tolerance": "exact", "max_abs_diff": "", "mismatches": int(bad.sum())})
    mismatches += [dict(zip(KEYS, r)) | {"field": col, "computed": c, "reference": e}
                   for r, c, e in zip(merged.loc[bad, KEYS].values, mine[bad], theirs[bad])]
diff_tbl = pd.DataFrame(diff_rows)
mismatch_tbl = pd.DataFrame(mismatches)
show("Field-by-field diff vs reference/dasher_scorecard.csv (54 Dashers)", diff_tbl)
show("Mismatching cells (empty = clean)", mismatch_tbl)
print("Scorecard diff:", "CLEAN" if mismatch_tbl.empty else f"{len(mismatch_tbl)} mismatching cells")

**Field-by-field diff vs reference/dasher_scorecard.csv (54 Dashers)**

,field,tolerance,max_abs_diff,mismatches
0,lifetime_deliveries,0,0.0,0
1,lifetime_star,0.001,0.0005,0
2,sample_deliveries,0,0,0
3,arrival_logged,0.001,0.0005,0
4,ontime_raw,0.001,0.00048,0
5,ontime_adj,0.001,0.00048,0
6,severe_late_rate,0.001,0.00049,0
7,shrunk_rating,0.001,0.00049,0
8,pts_reliability,0.001,0.0005,0
9,pts_quality,0.001,0.00047,0


**Mismatching cells (empty = clean)**

""


Scorecard diff: CLEAN


### 5.4 Lock: assert-based re-check
This cell fails loudly if any later change moves a score, gate or tier away from the reference.

In [42]:
def assert_matches_reference(scorecard, reference):
    m = scorecard.merge(reference, on=KEYS, how="outer", suffixes=("", "_ref"), indicator=True)
    assert (m["_merge"] == "both").all(), "Dasher set differs from reference"
    for col, tol in NUMERIC_TOL.items():
        gap = (m[col] - m[f"{col}_ref"]).abs()
        assert (gap <= tol).all(), f"{col}: max diff {gap.max():.4f} > {tol}"
    for col in EXACT:
        assert (m[col].fillna("").astype(str) == m[f"{col}_ref"].fillna("").astype(str)).all(), f"{col} differs"
    for market, size in TIER1_SIZE.items():
        assert ((m["MARKET_NAME"] == market) & (m["tier"] == "Tier 1")).sum() == size, f"{market} Tier 1 size"
    return True

assert assert_matches_reference(scorecard, reference)
print("assert_matches_reference: PASS")

assert_matches_reference: PASS


### 5.5 End-to-end traces: KC, Josh, Bret
Each trace walks from the ASAP order counts through every component to the gate and tier decision. The order-level
view shows only the orders whose late status changes under the wait adjustment (plus a count of the rest).

In [43]:
def trace(market, name):
    row = scorecard[(scorecard["MARKET_NAME"] == market) & (scorecard["DASHER"] == name)].iloc[0]
    orders = asap[(asap["MARKET_NAME"] == market) & (asap["DASHER"] == name)]
    all_orders = df[(df["MARKET_NAME"] == market) & (df["DASHER"] == name)]
    steps = [
        ("ASAP orders (asap_n)", len(orders)),
        ("  late vs quoted (raw)", int(orders["is_late"].sum())),
        ("  late after wait adjustment", int(orders["is_late_adj"].sum())),
        ("  severe (adjusted > 15 min)", int(orders["severe"].sum())),
        ("ontime_raw = 1 - late_raw / asap_n", round(row["ontime_raw"], 4)),
        ("ontime_adj = 1 - late_adj / asap_n", round(row["ontime_adj"], 4)),
        ("severe_late_rate = severe / asap_n", round(row["severe_late_rate"], 4)),
        ("lifetime deliveries (n)", row["lifetime_deliveries"]),
        ("lifetime star", row["lifetime_star"]),
        ("market mean star (dedup Dashers)", round(row["market_mean_star"], 4)),
        ("shrunk_rating = (n*star + 200*mean)/(n+200)", round(row["shrunk_rating"], 4)),
        ("all orders / with store arrival logged", f"{len(all_orders)} / {int(all_orders['arrival_logged'].sum())}"),
        ("arrival_share", round(row["arrival_logged"], 4)),
        ("pts_reliability = 50 * ontime_adj", round(row["pts_reliability"], 3)),
        ("pts_quality = 30 * shrunk_rating / 5", round(row["pts_quality"], 3)),
        ("pts_consistency = 10 * max(0, 1 - severe/0.30)", round(row["pts_consistency"], 3)),
        ("pts_professionalism = 10 * arrival_share", round(row["pts_professionalism"], 3)),
        ("selection_score", round(row["selection_score"], 3)),
        ("vehicles / widget_capable", f"{row['vehicles']} / {row['widget_capable']}"),
        (f"lifetime >= {GATE_LIFETIME_MIN[market]}", bool(row["lifetime_deliveries"] >= GATE_LIFETIME_MIN[market])),
        (f"asap_n >= {GATE_ASAP_MIN[market]}", bool(row["asap_n"] >= GATE_ASAP_MIN[market])),
        ("passes_gates", bool(row["passes_gates"])),
        ("fraud flags / fraud_review", f"{int(row['fraud_flags'])} / {row['fraud_review']}"),
        ("tier", row["tier"]),
    ]
    show(f"Trace: {name} ({market})", pd.DataFrame(steps, columns=["step", "value"]).set_index("step"))
    changed = orders[orders["is_late"] != orders["is_late_adj"]]
    show(f"{name}: orders whose late status changes under the wait adjustment "
         f"({len(changed)} of {len(orders)}; the other {len(orders) - len(changed)} keep their raw status)",
         changed[["DELIVERY_NAME", "late_min", "wait_min", "excess_wait", "late_min_adj", "is_late", "is_late_adj", "severe"]]
         .round(2).reset_index(drop=True))

trace("Dashattan", "KC")
trace("Doorlanta", "Josh")
trace("Dashattan", "Bret")

**Trace: KC (Dashattan)**

,value
step,
ASAP orders (asap_n),138
late vs quoted (raw),15
late after wait adjustment,10
severe (adjusted > 15 min),2
ontime_raw = 1 - late_raw / asap_n,0.8913
ontime_adj = 1 - late_adj / asap_n,0.9275
severe_late_rate = severe / asap_n,0.0145
lifetime deliveries (n),6580.0
lifetime star,4.54


**KC: orders whose late status changes under the wait adjustment (5 of 138; the other 133 keep their raw status)**

,DELIVERY_NAME,late_min,wait_min,excess_wait,late_min_adj,is_late,is_late_adj,severe
0,1385.0,0.48,28.67,18.67,-18.18,True,False,False
1,119.0,26.96,62.28,52.28,-25.32,True,False,False
2,258.0,2.19,12.57,2.57,-0.37,True,False,False
3,540.0,3.00,19.22,9.22,-6.21,True,False,False
4,123.0,42.46,61.40,51.40,-8.94,True,False,False


**Trace: Josh (Doorlanta)**

,value
step,
ASAP orders (asap_n),20
late vs quoted (raw),4
late after wait adjustment,1
severe (adjusted > 15 min),0
ontime_raw = 1 - late_raw / asap_n,0.8
ontime_adj = 1 - late_adj / asap_n,0.95
severe_late_rate = severe / asap_n,0.0
lifetime deliveries (n),1226.0
lifetime star,4.77


... (2 outputs left out)

**Trace: Bret (Dashattan)**

,value
step,
ASAP orders (asap_n),17
late vs quoted (raw),11
late after wait adjustment,8
severe (adjusted > 15 min),5
ontime_raw = 1 - late_raw / asap_n,0.3529
ontime_adj = 1 - late_adj / asap_n,0.5294
severe_late_rate = severe / asap_n,0.2941
lifetime deliveries (n),773.0
lifetime star,3.82


**Bret: orders whose late status changes under the wait adjustment (3 of 17; the other 14 keep their raw status)**

,DELIVERY_NAME,late_min,wait_min,excess_wait,late_min_adj,is_late,is_late_adj,severe
0,827.0,27.23,43.03,33.03,-5.81,True,False,False
1,851.0,4.84,18.22,8.22,-3.38,True,False,False
2,539.0,12.69,29.58,19.58,-6.90,True,False,False


### 5.6 Josh's rank: 7 → 2
`KEY_NUMBERS` says the wait adjustment moves Josh from rank 7 to rank 2. Two readings are checked among Doorlanta
gate-passers: rank by score with raw on-time in the Reliability component vs rank by the spec score, and rank by the
on-time rate itself.

In [44]:
PHASE3_RECON_START = len(recon)
door = scorecard[(scorecard["MARKET_NAME"] == "Doorlanta") & scorecard["passes_gates"]].copy()
door["score_with_raw_ontime"] = door["selection_score"] - door["pts_reliability"] + 50 * door["ontime_raw"]
rank_tbl = pd.DataFrame({
    "rank by score (raw on-time in Reliability)": door["score_with_raw_ontime"].rank(ascending=False, method="min"),
    "rank by score (spec, adjusted)": door["selection_score"].rank(ascending=False, method="min"),
    "rank by ontime_raw": door["ontime_raw"].rank(ascending=False, method="min"),
    "rank by ontime_adj": door["ontime_adj"].rank(ascending=False, method="min"),
})
rank_tbl.index = door["DASHER"].values
rank_tbl = rank_tbl.astype(int).sort_values("rank by score (spec, adjusted)")
show("Doorlanta gate-passers: ranks under each reading", rank_tbl)
josh = rank_tbl.loc["Josh"]
score_reading = (int(josh["rank by score (raw on-time in Reliability)"]), int(josh["rank by score (spec, adjusted)"]))
ontime_reading = (int(josh["rank by ontime_raw"]), int(josh["rank by ontime_adj"]))
print("Josh by score:", score_reading, "| Josh by on-time rate:", ontime_reading)
register("Q1", "Josh rank 7 -> 2 (score with raw vs adjusted on-time)", 1, int(score_reading == (7, 2)), 0,
         f"score reading gives {score_reading}; on-time-rate reading gives {ontime_reading}; "
         "human decision 2026-11-02: deck to be corrected to 8 -> 2 (key left as is)")

**Doorlanta gate-passers: ranks under each reading**

,rank by score (raw on-time in Reliability),"rank by score (spec, adjusted)",rank by ontime_raw,rank by ontime_adj
Ajay,1,1,1,1
Josh,8,2,9,3
Jagar,2,3,2,2
Elon,3,4,3,4
Abhai,4,5,4,5
Hanna,5,6,6,8
Katelynn,6,7,5,6
Anisha,7,8,8,9
Cate,9,9,7,7
Hersh,11,10,12,10


Josh by score:

 (8, 2) | Josh by on-time rate: (9, 3)


### 5.7 Conditional bike cohort (display only)
Dashers who fail **only** the vehicle gate: bicycle-only, but meeting their market's lifetime and sample thresholds.
Ranked by the same score; the top 4 are highlighted. This changes nothing in the scorecard, gates or tiers above.

In [45]:
mk = scorecard["MARKET_NAME"]
bike_cohort = scorecard[
    ~scorecard["widget_capable"]
    & (scorecard["lifetime_deliveries"] >= mk.map(GATE_LIFETIME_MIN))
    & (scorecard["asap_n"].fillna(0) >= mk.map(GATE_ASAP_MIN))
].sort_values("selection_score", ascending=False).reset_index(drop=True)
bike_cohort["top_4"] = bike_cohort.index < 4
show("Conditional bike cohort: fails only the vehicle gate, ranked by score",
     bike_cohort[["MARKET_NAME", "DASHER", "vehicles", "lifetime_deliveries", "asap_n", "ontime_adj", "shrunk_rating",
                  "severe_late_rate", "arrival_logged", "selection_score", "fraud_review", "top_4"]].round(3))
print("Top 4:", bike_cohort.loc[bike_cohort["top_4"], "DASHER"].tolist(), "(CLAUDE.md expects Ron, Zak, Bunje, Rojan)")

**Conditional bike cohort: fails only the vehicle gate, ranked by score**

,MARKET_NAME,DASHER,vehicles,lifetime_deliveries,asap_n,ontime_adj,shrunk_rating,severe_late_rate,arrival_logged,selection_score,fraud_review,top_4
0,Dashattan,Ron,bicycle,2618.0,24,1.000,4.554,0.000,0.962,96.938,False,True
1,Dashattan,Zak,bicycle,905.0,6,1.000,4.601,0.000,0.833,95.941,False,True
2,Dashattan,Bunje,bicycle,1579.0,39,0.949,4.674,0.000,1.000,95.482,True,True
3,Dashattan,Rojan,bicycle,788.0,62,0.919,4.558,0.032,0.897,91.212,False,True
4,Dashattan,Awstin,bicycle,1403.0,8,0.875,4.496,0.000,0.875,89.479,False,False
5,Dashattan,Sean,bicycle,522.0,44,0.864,4.507,0.045,0.959,88.298,False,False
6,Dashattan,J,bicycle,1027.0,32,0.781,4.579,0.031,0.938,84.870,False,False
7,Dashattan,Mike,bicycle,736.0,27,0.815,4.596,0.074,0.889,84.738,False,False
8,Dashattan,Etan,bicycle,228.0,8,0.750,4.407,0.000,1.000,83.945,False,False
9,Dashattan,Rachael,bicycle,278.0,6,0.833,4.471,0.167,1.000,82.935,False,False


Top 4: ['Ron', 'Zak', 'Bunje', 'Rojan'] (CLAUDE.md expects Ron, Zak, Bunje, Rojan)


### 5.8 Uniform-strict reconciliation (moved from Phase 2)
Definition (CLAUDE.md): widget-capable vehicle AND lifetime ≥ 300 AND asap_n ≥ 10 AND severe-late rate ≤ 10% AND
adjusted on-time ≥ 85%; missing severe-late counts as fail, missing on-time as 0. Expect 1 / 8.

In [46]:
scorecard["selected_uniform_strict"] = (
    scorecard["widget_capable"]
    & (scorecard["lifetime_deliveries"] >= 300)
    & (scorecard["asap_n"].fillna(0) >= 10)
    & (scorecard["severe_late_rate"].fillna(1.0) <= 0.10)
    & (scorecard["ontime_adj"].fillna(0.0) >= 0.85)
)
uniform_strict_counts = scorecard.groupby("MARKET_NAME")["selected_uniform_strict"].sum()
show("Uniform-strict qualifiers", scorecard[scorecard["selected_uniform_strict"]]
     [["MARKET_NAME", "DASHER", "lifetime_deliveries", "asap_n", "severe_late_rate", "ontime_adj", "selection_score"]]
     .round(3).reset_index(drop=True))
ref_us = reference.set_index(KEYS)["selected_uniform_strict"]
mine_us = scorecard.set_index(KEYS)["selected_uniform_strict"].reindex(ref_us.index)
print("Dashers where uniform-strict flag differs from reference:", int((mine_us != ref_us).sum()))
register("Q1", "Uniform strict qualifiers, Dashattan", 1, uniform_strict_counts["Dashattan"], 0, "full definition, Phase 3")
register("Q1", "Uniform strict qualifiers, Doorlanta", 8, uniform_strict_counts["Doorlanta"], 0, "full definition, Phase 3")
recon_tbl = pd.DataFrame(recon)
show("Phase 3 reconciliation rows (appended to the registry; full table re-exported below)",
     recon_tbl.iloc[PHASE3_RECON_START:])

**Uniform-strict qualifiers**

,MARKET_NAME,DASHER,lifetime_deliveries,asap_n,severe_late_rate,ontime_adj,selection_score
0,Dashattan,KC,6580.0,138,0.014,0.928,92.492
1,Doorlanta,Abhai,947.0,33,0.000,0.909,91.079
2,Doorlanta,Ajay,726.0,11,0.000,1.000,97.443
3,Doorlanta,Cate,7039.0,70,0.043,0.886,86.619
4,Doorlanta,Elon,604.0,76,0.000,0.947,93.820
5,Doorlanta,Hanna,2899.0,39,0.026,0.872,90.682
6,Doorlanta,Jagar,5138.0,46,0.022,0.957,95.173
7,Doorlanta,Josh,1226.0,20,0.000,0.950,95.490
8,Doorlanta,Katelynn,3264.0,61,0.016,0.902,90.509


Dashers where uniform-strict flag differs from reference: 0


**Phase 3 reconciliation rows (appended to the registry; full table re-exported below)**

,section,metric,expected,computed,tol,match,note
63,Q1,Josh rank 7 -> 2 (score with raw vs adjusted on-time),1.0,0.0,0.0,NO,"score reading gives (8, 2); on-time-rate reading gives (9, 3); human decision 2026-11-02: deck to be corrected to 8 -> 2 (key left as is)"
64,Q1,"Uniform strict qualifiers, Dashattan",1.0,1.0,0.0,yes,"full definition, Phase 3"
65,Q1,"Uniform strict qualifiers, Doorlanta",8.0,8.0,0.0,yes,"full definition, Phase 3"


### 5.9 Chart data for the deck (Phase 3 visuals)
**What this does:** the two deck visuals that need Phase 3 quantities: the eligibility comparison (city-calibrated gates vs
the uniform-strict criteria) and the per-Dasher inputs behind the selection bubble chart. Same definitions as above.

**What it feeds:** `viz_eligibility` and `viz_scatter` in `export_03_scorecard.xlsx`.

In [47]:
eligibility_counts = pd.DataFrame({
    "city-calibrated gates (lifetime >= 100/300, asap_n >= 5/10, widget-capable)": scorecard.groupby("MARKET_NAME")["passes_gates"].sum(),
    "uniform strict (lifetime >= 300, asap_n >= 10, severe <= 10%, on-time >= 85%, widget-capable)": uniform_strict_counts,
    "Tier 1 (top 4 / top 10 gate-passers by score)": scorecard.groupby("MARKET_NAME")["tier"].apply(lambda s: (s == "Tier 1").sum()),
}).T
show("viz_eligibility · Dashers qualifying per city under each ruler", eligibility_counts)

scatter_inputs = (scorecard[["MARKET_NAME", "DASHER", "ontime_adj", "shrunk_rating", "asap_n", "widget_capable",
                             "passes_gates", "tier", "selection_score"]]
                  .sort_values(["MARKET_NAME", "selection_score"], ascending=[True, False]).round(3).reset_index(drop=True))
show("viz_scatter · Per-Dasher bubble-chart inputs (all 54 Dashers)", scatter_inputs)

**viz_eligibility · Dashers qualifying per city under each ruler**

MARKET_NAME,Dashattan,Doorlanta
"city-calibrated gates (lifetime >= 100/300, asap_n >= 5/10, widget-capable)",5,15
"uniform strict (lifetime >= 300, asap_n >= 10, severe <= 10%, on-time >= 85%, widget-capable)",1,8
Tier 1 (top 4 / top 10 gate-passers by score),4,10


**viz_scatter · Per-Dasher bubble-chart inputs (all 54 Dashers)**

,MARKET_NAME,DASHER,ontime_adj,shrunk_rating,asap_n,widget_capable,passes_gates,tier,selection_score
0,Dashattan,Barry,1.000,4.640,3,False,False,None,97.838
1,Dashattan,Cass,1.000,4.490,3,False,False,None,96.942
2,Dashattan,Ron,1.000,4.554,24,False,False,None,96.938
3,Dashattan,Jim,1.000,4.463,2,False,False,None,96.780
4,Dashattan,Zak,1.000,4.601,6,False,False,None,95.941
5,Dashattan,Rich,1.000,4.320,1,True,False,None,95.923
6,Dashattan,Bunje,0.949,4.674,39,False,False,None,95.482
7,Dashattan,Brian,1.000,4.788,2,False,False,None,95.396
8,Dashattan,KC,0.928,4.538,138,True,True,Tier 1,92.492
9,Dashattan,Rojan,0.919,4.558,62,False,False,None,91.212


### 5.10 Acceptance and export
`export_03_scorecard.xlsx` carries every `export_02` sheet (reconciliation now includes the Phase 3 rows), `dasher_scorecard` in the reference column order, and the two Phase 3 viz sheets.

In [48]:
assert mismatch_tbl.empty, "scorecard diff not clean"
assert (uniform_strict_counts == pd.Series({"Dashattan": 1, "Doorlanta": 8})).all(), uniform_strict_counts.to_dict()
assert set(bike_cohort.loc[bike_cohort["top_4"], "DASHER"]) == {"Ron", "Zak", "Bunje", "Rojan"}, "bike cohort top 4"
print("Phase 3 acceptance: PASS")

scorecard_export = scorecard[list(reference.columns)]
plain_sheets_03 = dict(plain_sheets, reconciliation=(recon_tbl, False), dasher_scorecard=(scorecard_export, False))
viz_sheets_03 = dict(viz_sheets,
    viz_eligibility=[("Qualification counts per city: city-calibrated gates vs uniform-strict criteria", eligibility_counts, True)],
    viz_scatter=[("Per-Dasher bubble-chart inputs: adjusted on-time (ASAP) x shrunk rating, bubble = asap_n", scatter_inputs, False)])
show("wrote export_03_scorecard.xlsx", write_workbook("export_03_scorecard.xlsx", plain_sheets_03, viz_sheets_03))

Phase 3 acceptance: PASS


**wrote export_03_scorecard.xlsx**

,data rows
cleaned_deliveries,1200
city_summary,2
reconciliation,66
dasher_scorecard,54
viz_lateness_attribution,7
viz_promise_gap,2
viz_vehicle_mix,10
viz_hourly_demand,17
viz_batching,4
viz_wait_adjustment,20


## 6. Q2 sizing

*Folded.* The Q2 inputs that come from the data are the hourly demand profile and the two peak windows in **§3.3**
(exported as `viz_hourly_demand`). The density figures and the satchel sizing chain (1,000 → 350 → 58 → 76) are external /
assumption numbers, shown and labelled in **§3.4** and **§4.9** and exported as `viz_sizing_funnel`. No separate computation here.

## 7. Q3 baselines

*Folded.* The Q3 baselines — adjusted late rate (ASAP), long-merchant-hold share, low-rating share — are computed in
**§4.7** and exported as `viz_q3_baselines`. The evidence behind the Q3 launch rules is in **§4.1–4.2** (promise gap, scheduled
orders) and **§4.5** (batching). Targets are the human's and are not in the notebook.

## 8. Exploratory sweep

**What this section does:** the timeboxed sweep from `CLAUDE.md` Phase 4, items 14–33. Each item computes one small
table and gets exactly one verdict line: *nothing*, or *candidate* with the reason. No chart polishing, no new metrics;
every rate uses the Phase 1 definitions. Lateness tables use all orders unless the title says ASAP.

**What it feeds:** the findings summary at the end of the section lists the candidates for the human to promote into the
deck or discard. Nothing here changes a metric, gate or tier.

In [49]:
ex = df.copy()                      # working copy; nothing in this section writes back to df
pct = lambda s: round(100 * s.mean(), 1)                                   # share of True, in %
low_rating_pct = lambda s: round(100 * (s <= 3).sum() / s.notna().sum(), 1) if s.notna().any() else np.nan
print("sweep population:", len(ex), "orders |", len(asap), "ASAP")

sweep population: 1200 orders | 1126 ASAP


#### 14. Subtotal bands vs lateness and rating

In [50]:
# item 14 · subtotal bands (SUBTOTAL is cents) vs late rates and rating, all orders
ex = df.copy()
ex["subtotal_usd"] = ex["SUBTOTAL"] / 100
ex["subtotal_band"] = pd.cut(ex["subtotal_usd"], [0, 10, 25, 50, 10_000], right=False,   # plan-doc bands
                             labels=["$0–10", "$10–25", "$25–50", "$50+"])
t14 = ex.groupby("subtotal_band", observed=True).agg(
    orders=("is_late", "size"), raw_late_pct=("is_late", pct), adj_late_pct=("is_late_adj", pct),
    rated=("DELIVERY_RATING", "count"), mean_rating=("DELIVERY_RATING", "mean"), low_rating_pct=("DELIVERY_RATING", low_rating_pct),
).round(2)
show("14 · Late rate and rating by subtotal band (all orders; plan-doc bands)", t14)

**14 · Late rate and rating by subtotal band (all orders; plan-doc bands)**

,orders,raw_late_pct,adj_late_pct,rated,mean_rating,low_rating_pct
subtotal_band,,,,,,
$0–10,80,16.2,12.5,44,4.59,11.4
$10–25,584,16.1,12.3,296,4.57,10.8
$25–50,441,18.8,14.5,198,4.56,12.1
$50+,95,21.1,15.8,46,4.41,17.4


**candidate (weak):** with the plan-doc bands, late rate rises with order value (raw 16% under $25 → 19% at $25–50 → 21% at $50+; adjusted 12.3% → 14.5% → 15.8%) and $50+ orders have the highest low-rating share (17% vs 11–12%). $50+ is only 95 orders, but widgets are high-value.

#### 15. Item count vs merchant wait

In [51]:
# item 15 · item count vs merchant wait, all orders (long hold = wait > 10, missing wait counts as not long)
ex["item_band"] = pd.cut(ex["TOTAL_ITEM_COUNT"], [1, 2, 3, 6, 11, 1000], right=False, labels=["1", "2", "3–5", "6–10", "11+"])
t15 = ex.groupby("item_band", observed=True).agg(
    orders=("wait_min", "size"), wait_missing_pct=("wait_min", lambda s: round(100 * s.isna().mean(), 1)),
    median_wait_min=("wait_min", "median"), p90_wait_min=("wait_min", lambda s: s.quantile(0.9)),
    long_hold_pct=("wait_min", lambda s: round(100 * (s > 10).mean(), 1)),
).round(2)
show("15 · Merchant wait by item-count band (all orders)", t15)

**15 · Merchant wait by item-count band (all orders)**

,orders,wait_missing_pct,median_wait_min,p90_wait_min,long_hold_pct
item_band,,,,,
1,260,13.5,4.92,16.65,17.3
2,341,17.3,6.21,19.86,24.9
3–5,441,15.9,8.00,23.02,34.7
6–10,122,19.7,10.72,29.05,43.4
11+,36,8.3,13.35,40.50,63.9


**candidate:** merchant wait climbs steadily with basket size (median 4.9 min at 1 item → 13.4 min at 11+; long-hold share 17% → 64%). Basket size, not the Dasher, drives holds — relevant to the merchant-hold attribution and to small widget orders.

#### 16. Subtotal by city

In [52]:
# item 16 · subtotal distribution by city (dollars)
t16 = ex.groupby("MARKET_NAME")["subtotal_usd"].describe(percentiles=[0.25, 0.5, 0.75, 0.9])[["count", "mean", "25%", "50%", "75%", "90%", "max"]].round(2)
t16["share >= $50 %"] = ex.groupby("MARKET_NAME")["subtotal_usd"].apply(lambda s: round(100 * (s >= 50).mean(), 1))
show("16 · Subtotal (USD) by city", t16)

**16 · Subtotal (USD) by city**

,count,mean,25%,50%,75%,90%,max,share >= $50 %
MARKET_NAME,,,,,,,,
Dashattan,600.0,24.05,15.0,20.0,30.0,45.0,75.0,6.5
Doorlanta,600.0,25.62,15.0,20.0,35.0,45.0,75.0,9.3


**nothing:** subtotal distributions are near-identical by city (median $20 in both; mean $24 vs $26; 90th pct $45 in both).

#### 17. Day of week

In [53]:
# item 17 · day of week (ACTIVE_DATE = local delivery date), all orders
DOW = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
ex["dow"] = pd.Categorical(ex["ACTIVE_DATE"].dt.day_name(), categories=DOW, ordered=True)
t17 = ex.groupby("dow", observed=True).agg(orders=("is_late", "size"), raw_late_pct=("is_late", pct), adj_late_pct=("is_late_adj", pct),
                                            mean_rating=("DELIVERY_RATING", "mean")).round(2)
t17 = t17.join(ex.groupby(["dow", "MARKET_NAME"], observed=True).size().unstack())
t17["days in window"] = ex.groupby("dow", observed=True)["ACTIVE_DATE"].nunique()
show("17 · Volume and late rate by day of week (June 1–13 covers each weekday once or twice)", t17)

**17 · Volume and late rate by day of week (June 1–13 covers each weekday once or twice)**

,orders,raw_late_pct,adj_late_pct,mean_rating,Dashattan,Doorlanta,days in window
dow,,,,,,,
Monday,98,22.4,18.4,4.50,31,67,1
Tuesday,170,19.4,16.5,4.51,86,84,2
Wednesday,171,19.9,16.4,4.64,84,87,2
Thursday,180,17.8,13.3,4.60,99,81,2
Friday,200,14.0,9.0,4.48,98,102,2
Saturday,251,17.9,14.3,4.48,140,111,2
Sunday,130,12.3,6.9,4.75,62,68,2


**nothing:** Sunday 12% vs Monday 22% raw late looks like a pattern, but each weekday appears only once or twice in a 13-day window and is confounded with the daily trend in item 18.

#### 18. Daily trend

In [54]:
# item 18 · daily trend, all orders
t18 = ex.groupby(ex["ACTIVE_DATE"].dt.date).agg(orders=("is_late", "size"), raw_late_pct=("is_late", pct), adj_late_pct=("is_late_adj", pct),
                                                 median_wait_min=("wait_min", "median")).round(2)
t18 = t18.join(ex.groupby([ex["ACTIVE_DATE"].dt.date, "MARKET_NAME"]).size().unstack())
t18.index.name = "date"
show("18 · Daily volume, late rate and median wait", t18)

**18 · Daily volume, late rate and median wait**

,orders,raw_late_pct,adj_late_pct,median_wait_min,Dashattan,Doorlanta
date,,,,,,
2055-06-01,109,17.4,14.7,6.53,43.0,66.0
2055-06-02,141,18.4,14.2,5.33,54.0,87.0
2055-06-03,144,14.6,10.4,7.52,63.0,81.0
2055-06-04,141,14.9,11.3,7.16,39.0,102.0
2055-06-05,186,16.7,14.0,6.68,75.0,111.0
2055-06-06,97,12.4,7.2,9.15,29.0,68.0
2055-06-07,98,22.4,18.4,5.93,31.0,67.0
2055-06-08,61,23.0,19.7,7.03,43.0,18.0
2055-06-09,30,26.7,26.7,7.71,30.0,NaN


**candidate (data-scope caveat):** Doorlanta has no deliveries after June 8 — its 600 orders span 8 days (~75/day), Dashattan's span 13 (~46/day). June 9–13 is Dashattan-only and thin (30–65 orders/day, late rate swinging 12–31%). Any 'June 1–13' or per-day-volume statement needs this caveat; it also empties the second half of item 23.

#### 19. Late rate by local hour

In [55]:
# item 19 · late rate by local hour of delivery, all orders (hours with < 20 orders marked thin)
t19 = ex.groupby("hour_local").agg(orders=("is_late", "size"), raw_late_pct=("is_late", pct), adj_late_pct=("is_late_adj", pct),
                                   long_hold_pct=("wait_min", lambda s: round(100 * (s > 10).mean(), 1))).round(1)
t19["thin (<20 orders)"] = t19["orders"] < 20
show("19 · Late rate by local hour", t19.T)

**19 · Late rate by local hour**

hour_local,0,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
orders,1,1,12,31,75,118,104,76,50,57,63,114,158,150,123,56,11
raw_late_pct,0.0,100.0,41.7,38.7,10.7,19.5,17.3,14.5,16.0,21.1,22.2,14.0,14.6,16.7,15.4,23.2,18.2
adj_late_pct,0.0,100.0,41.7,22.6,5.3,11.0,13.5,11.8,16.0,21.1,14.3,9.6,12.0,15.3,12.2,17.9,9.1
long_hold_pct,0.0,0.0,8.3,38.7,38.7,34.7,31.7,27.6,22.0,14.0,30.2,28.1,25.9,30.0,33.3,35.7,45.5
thin (<20 orders),True,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True


**candidate (weak):** adjusted late rate is lowest at the lunch peak (11:00 5%, 12:00 11%) and highest mid-afternoon (15–17h 14–21%) and at 22h (18%); the dinner peak 18–21h sits at 10–15%. Lateness is not peak-driven. Hours before 10 are thin.

#### 20. Assignment lag by local hour

In [56]:
# item 20 · assignment lag by local hour, ASAP only
t20 = asap.groupby("hour_local")["assign_lag_min"].agg(orders="size", median="median", p90=lambda s: s.quantile(0.9), max="max").round(1)
show("20 · Assignment lag (minutes) by local hour, ASAP orders", t20.T)

**20 · Assignment lag (minutes) by local hour, ASAP orders**

hour_local,0,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
orders,1.0,8.0,24.0,52.0,105.0,102.0,74.0,49.0,55.0,59.0,110.0,151.0,148.0,121.0,56.0,11.0
median,24.8,4.8,6.0,2.5,3.3,3.5,4.7,6.2,4.7,3.2,2.5,4.7,4.3,5.0,6.0,3.6
p90,24.8,20.2,11.3,13.6,13.1,21.0,15.8,25.6,20.7,14.6,8.8,14.4,17.6,18.1,20.1,38.0
max,24.8,37.1,24.9,29.5,44.3,95.9,42.3,51.2,59.8,22.0,28.6,89.9,44.7,131.0,69.1,54.7


**nothing:** median assignment lag is 2.5–6 min in every hour with ≥ 20 orders; the p90 tail (9–26 min) does not follow the demand peaks.

#### 21. Bike vs car within Dashattan

In [57]:
# item 21 · bike vs car within Dashattan: speed segments and late rates, all Dashattan orders
dash_all = ex[ex["MARKET_NAME"] == "Dashattan"]
t21 = dash_all.groupby("VEHICLE").agg(
    orders=("is_late", "size"),
    median_travel_to_customer_min=("RESTAURANT_TO_CUSTOMER_DURATION", lambda s: s.median() / 60),
    p90_travel_to_customer_min=("RESTAURANT_TO_CUSTOMER_DURATION", lambda s: s.quantile(0.9) / 60),
    median_travel_to_store_min=("DASHER_TO_RESTAURANT_DURATION", lambda s: s.median() / 60),
    median_confirm_to_deliver_min=("CONFIRM_TO_DELIVER_DURATION", lambda s: s.median() / 60),
    raw_late_pct=("is_late", pct), adj_late_pct=("is_late_adj", pct), mean_rating=("DELIVERY_RATING", "mean"),
).round(2)
show("21 · Dashattan: bicycle vs car", t21.T)

**21 · Dashattan: bicycle vs car**

VEHICLE,bicycle,car
orders,369.00,231.00
median_travel_to_customer_min,12.05,12.12
p90_travel_to_customer_min,24.08,22.55
median_travel_to_store_min,10.65,8.92
median_confirm_to_deliver_min,32.73,32.12
raw_late_pct,18.70,20.30
adj_late_pct,13.60,15.60
mean_rating,4.58,4.52


**candidate:** within Dashattan, bicycles match cars on the customer leg (median 12.1 vs 12.1 min; p90 24.1 vs 22.6) and have a lower adjusted late rate (13.6% vs 15.6%); cars are faster only to the store (8.9 vs 10.7 min). Speed is not the reason to exclude bikes — carrying capacity is.

#### 22. Dasher utilization

In [58]:
# item 22 · Dasher utilization: orders per active day (days with >= 1 delivery in the window), all orders
util = ex.groupby(["MARKET_NAME", "DASHER"]).agg(orders=("DELIVERY_NAME", "size"), active_days=("ACTIVE_DATE", "nunique"))
util["orders_per_active_day"] = (util["orders"] / util["active_days"]).round(2)
t22 = util.groupby(level="MARKET_NAME").agg(dashers=("orders", "size"), median_orders=("orders", "median"), max_orders=("orders", "max"),
                                            median_active_days=("active_days", "median"),
                                            median_per_day=("orders_per_active_day", "median"), max_per_day=("orders_per_active_day", "max"))
show("22 · Utilization summary per market", t22)
show("22 · Top 8 Dashers by orders per active day", util.sort_values("orders_per_active_day", ascending=False).head(8))
t22b = util["orders_per_active_day"].describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(2).to_frame("orders per active day").T
t22b["Dashers at >= 6 per active day"] = int((util["orders_per_active_day"] >= 6).sum())
show("22 · Orders per active day across all 54 Dashers vs the 6-per-Dasher-day sizing assumption", t22b)

**22 · Utilization summary per market**

,dashers,median_orders,max_orders,median_active_days,median_per_day,max_per_day
MARKET_NAME,,,,,,
Dashattan,24,9.0,143,3.0,3.415,13.0
Doorlanta,30,12.0,82,3.0,3.900,16.4


**22 · Top 8 Dashers by orders per active day**

orders  active_days  orders_per_active_day
MARKET_NAME DASHER                                              
Doorlanta   Elon          82            5                  16.40
Dashattan   KC           143           11                  13.00
Doorlanta   Ajay          11            1                  11.00
            Cate          72            7                  10.29
            Katelynn      64            7                   9.14
            Jagar         50            6                   8.33
Dashattan   Bill           8            1                   8.00
            Samm          48            7                   6.86

**22 · Orders per active day across all 54 Dashers vs the 6-per-Dasher-day sizing assumption**

,count,mean,std,min,25%,50%,75%,90%,max,Dashers at >= 6 per active day
orders per active day,54.0,4.65,3.17,1.0,3.0,3.65,6.5,8.23,16.4,18


**candidate:** utilization is concentrated (KC 13 orders per active day over 11 days, Elon 16 over 5; the median Dasher does 9–12 orders over 3 active days). Against the sizing assumption: the median Dasher completes 3.65 orders per active day and only 18 of 54 reach 6, so 6 per Dasher-day sits at roughly the 75th percentile of observed food throughput.

#### 23. Half-window consistency

In [59]:
# item 23 · half-window consistency: adjusted on-time per gate-passing Dasher, June 1–7 vs June 8–13 (ASAP only)
half = asap.assign(half=np.where(asap["ACTIVE_DATE"].dt.day <= 7, "Jun 1–7", "Jun 8–13"))
hw = half.groupby(["MARKET_NAME", "DASHER", "half"]).agg(n=("is_late_adj", "size"), ontime_adj=("is_late_adj", lambda s: 1 - s.mean())).unstack("half")
hw.columns = [f"{a}_{b}" for a, b in hw.columns]
hw = hw.reset_index().merge(scorecard[["MARKET_NAME", "DASHER", "passes_gates", "tier"]])
t23 = hw[hw["passes_gates"]].drop(columns="passes_gates").round(3).sort_values(["MARKET_NAME", "DASHER"]).reset_index(drop=True)
r_halves = t23[["ontime_adj_Jun 1–7", "ontime_adj_Jun 8–13"]].corr().iloc[0, 1]
show(f"23 · Gate-passers: adjusted on-time in each half of the window (Pearson r across Dashers = {r_halves:.2f})", t23)

**23 · Gate-passers: adjusted on-time in each half of the window (Pearson r across Dashers = 0.47)**

,MARKET_NAME,DASHER,n_Jun 1–7,n_Jun 8–13,ontime_adj_Jun 1–7,ontime_adj_Jun 8–13,tier
0,Dashattan,Bill,8.0,NaN,0.750,NaN,Tier 1
1,Dashattan,Bret,14.0,3.0,0.500,0.667,Tier 2
2,Dashattan,Carlee,NaN,6.0,NaN,0.667,Tier 1
3,Dashattan,KC,79.0,59.0,0.924,0.932,Tier 1
4,Dashattan,Samm,41.0,3.0,0.854,0.667,Tier 1
5,Doorlanta,Abhai,33.0,NaN,0.909,NaN,Tier 1
6,Doorlanta,Ajay,11.0,NaN,1.000,NaN,Tier 1
7,Doorlanta,Anisha,13.0,NaN,0.846,NaN,Tier 1
8,Doorlanta,Ash,10.0,2.0,0.700,1.000,Tier 2
9,Doorlanta,Cate,67.0,3.0,0.881,1.000,Tier 1


**nothing (cannot assess):** only 9 gate-passers have orders in both halves, and because Doorlanta's data ends June 8 the second-half samples are 1–3 orders for all but KC (0.92 → 0.93). r = 0.47 on 9 Dashers is not evidence either way.

#### 24. Confirm-speed tail

In [60]:
# item 24 · confirm speed: seconds from assignment to Dasher confirmation, all orders
confirm_sec = (ex["DASHER_CONFIRMED_TIME"] - ex["DASHER_ASSIGNED_TIME"]).dt.total_seconds()
t24 = pd.DataFrame({
    "all": confirm_sec.describe(percentiles=[0.5, 0.9, 0.95, 0.99])[["count", "50%", "90%", "95%", "99%", "max"]],
    "Dashattan": confirm_sec[ex["MARKET_NAME"] == "Dashattan"].describe(percentiles=[0.5, 0.9, 0.95, 0.99])[["count", "50%", "90%", "95%", "99%", "max"]],
    "Doorlanta": confirm_sec[ex["MARKET_NAME"] == "Doorlanta"].describe(percentiles=[0.5, 0.9, 0.95, 0.99])[["count", "50%", "90%", "95%", "99%", "max"]],
}).round(1)
t24.loc["orders > 120 s"] = [(confirm_sec > 120).sum(), (confirm_sec[ex["MARKET_NAME"] == "Dashattan"] > 120).sum(), (confirm_sec[ex["MARKET_NAME"] == "Doorlanta"] > 120).sum()]
show("24 · Assignment-to-confirmation seconds", t24)

**24 · Assignment-to-confirmation seconds**

,all,Dashattan,Doorlanta
count,1200.0,600.0,600.0
50%,11.5,13.8,9.0
90%,36.3,38.1,33.3
95%,50.4,52.9,48.9
99%,89.7,96.1,80.5
max,136.0,111.1,136.0
orders > 120 s,1.0,0.0,1.0


**nothing:** confirmation is fast everywhere — median 11.5 s, p99 90 s, one order over two minutes.

#### 25. Rating response bias

In [61]:
# item 25 · who leaves a rating: response rate by adjusted late status and city, all orders
ex["rated"] = ex["DELIVERY_RATING"].notna()
t25 = ex.groupby(["MARKET_NAME", "is_late_adj"]).agg(orders=("rated", "size"), rated_pct=("rated", pct),
                                                     mean_rating_if_rated=("DELIVERY_RATING", "mean"),
                                                     low_rating_pct=("DELIVERY_RATING", low_rating_pct)).round(2)
show("25 · Rating response by late status", t25)

**25 · Rating response by late status**

orders  rated_pct  mean_rating_if_rated  low_rating_pct
MARKET_NAME is_late_adj                                                         
Dashattan   False           514       46.5                  4.63             9.2
            True             86       32.6                  3.93            25.0
Doorlanta   False           525       52.8                  4.60            11.2
            True             75       53.3                  4.22            22.5

**candidate:** rating response is biased in Dashattan — late deliveries are rated 33% of the time vs 47% for on-time ones, while Doorlanta rates both at 53%. Rated late orders average 3.9–4.2 vs 4.6, so Dashattan's rating baseline is flattered by non-response on late orders.

#### 26. Rating vs minutes late

In [62]:
# item 26 · rating vs adjusted minutes late, rated orders only
ex["late_band"] = pd.cut(ex["late_min_adj"], [-1e9, 0, 5, 15, 1e9], labels=["on time (<= 0)", "0–5 late", "5–15 late", "15+ late"])
t26 = ex.groupby("late_band", observed=True).agg(orders=("rated", "size"), rated_pct=("rated", pct), mean_rating=("DELIVERY_RATING", "mean"),
                                                 low_rating_pct=("DELIVERY_RATING", low_rating_pct)).round(2)
show("26 · Rating by adjusted lateness band", t26)

**26 · Rating by adjusted lateness band**

,orders,rated_pct,mean_rating,low_rating_pct
late_band,,,,
on time (<= 0),1039,49.7,4.61,10.3
0–5 late,45,28.9,4.69,7.7
5–15 late,65,47.7,4.45,19.4
15+ late,51,47.1,3.33,37.5


**candidate:** ratings barely move for 0–15 min of adjusted lateness (mean 4.45–4.69) and collapse beyond 15 min (mean 3.33, 38% low). Direct support for the 15-min severe-late threshold.

#### 27. Rating vs merchant wait

In [63]:
# item 27 · rating vs merchant wait (missing wait kept as its own band), rated orders only
ex["wait_band"] = pd.cut(ex["wait_min"], [0, 5, 10, 20, 1e9], right=False, labels=["0–5", "5–10", "10–20", "20+"]).astype(object).fillna("missing")
t27 = ex.groupby("wait_band").agg(orders=("rated", "size"), rated_pct=("rated", pct), mean_rating=("DELIVERY_RATING", "mean"),
                                  low_rating_pct=("DELIVERY_RATING", low_rating_pct), adj_late_pct=("is_late_adj", pct)).round(2)
show("27 · Rating by merchant-wait band (minutes)", t27.loc[["0–5", "5–10", "10–20", "20+", "missing"]])
ontime_only = ex[~ex["is_late_adj"]]
t27b = ontime_only.groupby("wait_band").agg(orders=("rated", "size"), rated_pct=("rated", pct), mean_rating=("DELIVERY_RATING", "mean"),
                                            low_rating_pct=("DELIVERY_RATING", low_rating_pct)).round(2)
show("27 · On-time (adjusted) deliveries only: rating by merchant-wait band", t27b.loc[["0–5", "5–10", "10–20", "20+", "missing"]])

**27 · Rating by merchant-wait band (minutes)**

,orders,rated_pct,mean_rating,low_rating_pct,adj_late_pct
wait_band,,,,,
0–5,367,46.9,4.68,8.7,13.6
5–10,281,48.4,4.47,13.2,11.7
10–20,235,48.1,4.55,10.6,11.5
20+,126,53.2,4.46,14.9,6.3
missing,191,50.3,4.52,14.6,22.5


**27 · On-time (adjusted) deliveries only: rating by merchant-wait band**

,orders,rated_pct,mean_rating,low_rating_pct
wait_band,,,,
0–5,317,48.6,4.71,7.8
5–10,248,48.0,4.53,11.8
10–20,208,50.0,4.56,10.6
20+,118,55.1,4.49,13.8
missing,148,50.0,4.73,9.5


**candidate:** on on-time deliveries only, long merchant holds still dent ratings — mean 4.71 at 0–5 min wait → 4.49 at 20+, low-rating share 7.8% → 13.8% — so the merchant metric matters for customer experience, not just lateness. Across all orders the band means are flat (4.46–4.68) because lateness confounds them. Orders with no logged wait are late 22.5% vs 6–14% when logged.

#### 28. Manual assignments

In [64]:
# item 28 · manually assigned orders vs the rest, all orders (assignment lag shown for ASAP only)
t28 = ex.groupby("MANUALLY_ASSIGNED").agg(orders=("is_late", "size"), Dashattan=("MARKET_NAME", lambda s: (s == "Dashattan").sum()),
                                          asap_pct=("ASAP", pct), raw_late_pct=("is_late", pct), adj_late_pct=("is_late_adj", pct),
                                          mean_rating=("DELIVERY_RATING", "mean")).round(2)
t28["median_assign_lag_min_asap"] = asap.groupby("MANUALLY_ASSIGNED")["assign_lag_min"].median().round(2)
t28.index = t28.index.map({0: "automatic", 1: "manual"})
show("28 · Manual vs automatic assignment", t28)
manual = ex[ex["MANUALLY_ASSIGNED"] == 1]
t28b = manual.groupby(["MARKET_NAME", "DASHER"]).agg(orders=("is_late", "size"), late_adj=("is_late_adj", "sum")).reset_index()
t28c = manual.groupby(manual["ACTIVE_DATE"].dt.date).size().rename("manual orders").to_frame().T
show("28 · Manual assignments: who (orders and adjusted-late count per Dasher)", t28b)
show("28 · Manual assignments: when", t28c)

**28 · Manual vs automatic assignment**

,orders,Dashattan,asap_pct,raw_late_pct,adj_late_pct,mean_rating,median_assign_lag_min_asap
MANUALLY_ASSIGNED,,,,,,,
automatic,1175,587,94.0,16.8,12.6,4.57,3.93
manual,25,13,84.0,52.0,52.0,3.82,9.75


**28 · Manual assignments: who (orders and adjusted-late count per Dasher)**

,MARKET_NAME,DASHER,orders,late_adj
0,Dashattan,Arya,1,0
1,Dashattan,Bret,1,1
2,Dashattan,J,1,1
3,Dashattan,Jennifer,1,1
4,Dashattan,KC,5,2
5,Dashattan,Rachael,1,1
6,Dashattan,Rojan,1,1
7,Dashattan,Samm,1,1
8,Dashattan,Sean,1,1
9,Doorlanta,Ash,1,1


**28 · Manual assignments: when**

ACTIVE_DATE,2055-06-01,2055-06-02,2055-06-03,2055-06-04,2055-06-05,2055-06-06,2055-06-07,2055-06-08
manual orders,7,4,4,1,2,1,2,4


**candidate:** the 25 manually assigned orders are late 52% (raw and adjusted) vs 13–17%, carry 2.5× the dispatch lag (median 9.8 vs 3.9 min) and lower ratings (3.8 vs 4.6). Who: 18 Dashers, mostly one order each (KC 5, 2 late). When: 15 of 25 fall on June 1–3 and none after June 8. Small n, but a clear dispatch-process flag.

#### 29. Fraud-row profile

In [65]:
# item 29 · the 9 fraud-flagged deliveries
t29 = ex.loc[ex["FRAUDULENT"] == 1, ["DELIVERY_NAME", "MARKET_NAME", "DASHER", "VEHICLE", "ASAP", "WAS_BATCHED", "subtotal_usd",
                                      "TOTAL_ITEM_COUNT", "DELIVERY_RATING", "late_min", "late_min_adj", "wait_min"]].round(1).reset_index(drop=True)
show("29 · Fraud-flagged deliveries", t29)
fraud = ex[ex["FRAUDULENT"] == 1]
t29b = pd.DataFrame({
    "fraud-flagged (9)": [fraud["subtotal_usd"].mean(), fraud["subtotal_usd"].median(), fraud["TOTAL_ITEM_COUNT"].mean(), fraud["TOTAL_ITEM_COUNT"].median()],
    "all orders": [ex["subtotal_usd"].mean(), ex["subtotal_usd"].median(), ex["TOTAL_ITEM_COUNT"].mean(), ex["TOTAL_ITEM_COUNT"].median()],
}, index=["mean subtotal $", "median subtotal $", "mean items", "median items"]).round(1)
show("29 · Fraud-flagged orders vs all orders: subtotal and item skew", t29b)

**29 · Fraud-flagged deliveries**

,DELIVERY_NAME,MARKET_NAME,DASHER,VEHICLE,ASAP,WAS_BATCHED,subtotal_usd,TOTAL_ITEM_COUNT,DELIVERY_RATING,late_min,late_min_adj,wait_min
0,1012.0,Dashattan,Bret,car,1.0,0.0,15.0,22.0,5.0,-16.4,-58.6,52.2
1,1448.0,Dashattan,KC,car,1.0,0.0,75.0,23.0,5.0,-20.6,-40.2,29.6
2,743.0,Doorlanta,Faud,car,1.0,0.0,55.0,5.0,NaN,-11.7,-11.7,6.7
3,320.0,Doorlanta,Hanna,car,1.0,0.0,40.0,7.0,5.0,-6.1,-17.1,21.0
4,977.0,Dashattan,Arya,bicycle,1.0,0.0,40.0,4.0,NaN,-6.5,-6.5,8.7
5,404.0,Dashattan,Samm,car,1.0,0.0,25.0,4.0,NaN,-15.4,-15.4,5.8
6,1344.0,Dashattan,Bunje,bicycle,1.0,1.0,25.0,4.0,1.0,-26.6,-40.2,23.5
7,308.0,Dashattan,KC,car,1.0,1.0,20.0,2.0,NaN,32.2,8.4,33.8
8,1.0,Doorlanta,Katelynn,car,0.0,0.0,20.0,5.0,5.0,-14.2,-14.2,2.9


**29 · Fraud-flagged orders vs all orders: subtotal and item skew**

,fraud-flagged (9),all orders
mean subtotal $,35.0,24.8
median subtotal $,25.0,20.0
mean items,8.4,3.4
median items,5.0,2.0


**candidate (weak):** no lateness or rating pattern (8 of 9 delivered early; ratings 5 where present except one 1★, Bunje), but the flagged orders skew to big baskets — mean 8.4 items vs 3.4 overall (median 5 vs 2), mean subtotal $35 vs $25. Supports review-not-exclude and, per the plan doc, basket-size monitoring as its own fraud guardrail.

#### 30. Estimate-error distribution

In [66]:
# item 30 · estimate error (actual - estimated, minutes) and quote error (actual - quoted) for ASAP orders
err = pd.DataFrame({"vs raw estimate": (asap["ACTUAL_DELIVERY_TIME"] - asap["ESTIMATED_DELIVERY_TIME"]).dt.total_seconds() / 60,
                    "vs quoted promise": asap["late_min"]})
t30 = err.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.8, 0.9, 0.95]).loc[["mean", "10%", "25%", "50%", "75%", "80%", "90%", "95%"]].round(2)
t30.loc["share > 0 (late) %"] = (100 * (err > 0).mean()).round(1)
show("30 · Delivery-time error distribution, ASAP orders (minutes; p80 vs raw estimate is the buffer that would cover 80%)", t30)

**30 · Delivery-time error distribution, ASAP orders (minutes; p80 vs raw estimate is the buffer that would cover 80%)**

,vs raw estimate,vs quoted promise
mean,4.67,-12.71
10%,-11.86,-31.46
25%,-5.72,-24.41
50%,1.77,-14.99
75%,12.00,-4.39
80%,15.16,-1.69
90%,24.24,7.88
95%,32.74,16.67
share > 0 (late) %,55.60,16.90


**candidate:** p80 of actual-minus-estimate is 15.2 min, so the ~17-min mean buffer covers roughly 80% of ASAP orders; covering 90% would take 24 min. Quantifies the promise logic and the 'why 17' question.

#### 31. Vehicle consistency

In [67]:
# item 31 · vehicle consistency per Dasher
veh_n = ex.groupby(["MARKET_NAME", "DASHER"])["VEHICLE"].nunique()
multi = veh_n[veh_n > 1]
show("31 · Dashers with more than one vehicle in the window", multi.rename("vehicles").to_frame())
t31 = (ex[ex["DASHER"].isin(multi.index.get_level_values("DASHER"))]
       .groupby(["DASHER", "VEHICLE"]).agg(orders=("is_late", "size"), raw_late_pct=("is_late", pct), adj_late_pct=("is_late_adj", pct),
                                           median_travel_to_customer_min=("RESTAURANT_TO_CUSTOMER_DURATION", lambda s: s.median() / 60)).round(2))
show("31 · Their orders by vehicle", t31)

**31 · Dashers with more than one vehicle in the window**

,,vehicles
MARKET_NAME,DASHER,
Doorlanta,Lauran,2


**31 · Their orders by vehicle**

orders  raw_late_pct  adj_late_pct  median_travel_to_customer_min
DASHER VEHICLE                                                                      
Lauran car             14          42.9          35.7                          16.33
       motorcycle       3           0.0           0.0                          17.65

**nothing:** Lauran is the only Dasher with two vehicles (14 car / 3 motorcycle orders); too few motorcycle orders to compare.

#### 32. IS_FIRST_DELIVERY check

In [68]:
# item 32 · IS_FIRST_DELIVERY rows and whether excluding them moves the city late rates
first = ex[ex["IS_FIRST_DELIVERY"] == 1]
show("32 · Rows flagged as first delivery", first[["DELIVERY_NAME", "MARKET_NAME", "DASHER", "NUM_DELIVERIES", "late_min", "late_min_adj", "DELIVERY_RATING"]].round(1).reset_index(drop=True))
t32 = pd.DataFrame({
    "raw late % (all rows)": ex.groupby("MARKET_NAME")["is_late"].apply(pct),
    "raw late % (excluding flagged)": ex[ex["IS_FIRST_DELIVERY"] == 0].groupby("MARKET_NAME")["is_late"].apply(pct),
    "adj late % (all rows)": ex.groupby("MARKET_NAME")["is_late_adj"].apply(pct),
    "adj late % (excluding flagged)": ex[ex["IS_FIRST_DELIVERY"] == 0].groupby("MARKET_NAME")["is_late_adj"].apply(pct),
})
show("32 · City late rates with and without the 7 flagged rows", t32)

**32 · Rows flagged as first delivery**

,DELIVERY_NAME,MARKET_NAME,DASHER,NUM_DELIVERIES,late_min,late_min_adj,DELIVERY_RATING
0,613.0,Doorlanta,Kev,144.0,-31.7,-31.7,5.0
1,1220.0,Dashattan,Awstin,1403.0,-17.4,-17.4,5.0
2,729.0,Doorlanta,Jarry,198.0,-3.7,-13.8,NaN
3,118.0,Doorlanta,Matthew,223.0,5.7,-8.2,NaN
4,1554.0,Dashattan,Dave,142.0,12.2,12.2,NaN
5,1111.0,Dashattan,Mike,736.0,5.4,-9.4,5.0
6,1141.0,Dashattan,Barry,418.0,-31.8,-38.6,5.0


**32 · City late rates with and without the 7 flagged rows**

,raw late % (all rows),raw late % (excluding flagged),adj late % (all rows),adj late % (excluding flagged)
MARKET_NAME,,,,
Dashattan,19.3,19.1,14.3,14.3
Doorlanta,15.7,15.6,12.5,12.6


**nothing:** the 7 flagged 'first deliveries' belong to Dashers with 142–1,403 lifetime deliveries, and excluding them moves city late rates by ≤ 0.2 pt. The flag is unreliable and harmless.

#### 33. Correlation sweep

In [69]:
# item 33 · Pearson correlations |r| > 0.2 across numeric fields (excluded columns left out); pairs that share a
# derivation (e.g. late_min -> late_min_adj, the duration segments -> confirm-to-deliver) are marked definitional
CORR_COLS = ["late_min", "late_min_adj", "is_late", "is_late_adj", "severe", "wait_min", "excess_wait", "assign_lag_min", "buffer_min",
             "hour_local", "SUBTOTAL", "TOTAL_ITEM_COUNT", "DELIVERY_RATING", "CONFIRM_TO_DELIVER_DURATION",
             "DASHER_TO_RESTAURANT_DURATION", "RESTAURANT_TO_CUSTOMER_DURATION", "WAS_BATCHED", "ASAP", "MANUALLY_ASSIGNED",
             "FRAUDULENT", "COMPOSITE_STAR_RATING", "NUM_DELIVERIES", "NUM_FIVE_STARS", "arrival_logged"]
DERIVATION_GROUPS = [
    {"late_min", "late_min_adj", "is_late", "is_late_adj", "severe", "excess_wait", "wait_min"},
    {"CONFIRM_TO_DELIVER_DURATION", "DASHER_TO_RESTAURANT_DURATION", "RESTAURANT_TO_CUSTOMER_DURATION", "wait_min", "excess_wait"},
    {"NUM_DELIVERIES", "NUM_FIVE_STARS"}, {"wait_min", "arrival_logged", "excess_wait"}, {"ASAP", "buffer_min", "assign_lag_min"},
]
cm = ex[CORR_COLS].astype(float).corr()
pairs = cm.where(np.triu(np.ones(cm.shape, dtype=bool), k=1)).stack().rename("r").reset_index()
pairs.columns = ["a", "b", "r"]
pairs["definitional"] = [any({a, b} <= g for g in DERIVATION_GROUPS) for a, b in zip(pairs["a"], pairs["b"])]
t33 = pairs[pairs["r"].abs() > 0.2].assign(abs_r=lambda t: t["r"].abs()).sort_values("abs_r", ascending=False).drop(columns="abs_r").round(3).reset_index(drop=True)
show(f"33 · Pairs with |r| > 0.2 ({len(t33)} of {len(pairs)}; non-definitional pairs are the ones to read)", t33)
PLAN_PAIRS = ["SUBTOTAL", "TOTAL_ITEM_COUNT", "hour_local", "WAS_BATCHED", "assign_lag_min", "wait_min"]
t33b = asap[["late_min"] + PLAN_PAIRS].astype(float).corr().loc[["late_min"], PLAN_PAIRS].round(3)
t33b.index = ["r with late_min (ASAP orders)"]
show("33 · The plan doc's six named pairs", t33b)

**33 · Pairs with |r| > 0.2 (47 of 576; non-definitional pairs are the ones to read)**

,a,b,r,definitional
0,NUM_DELIVERIES,NUM_FIVE_STARS,0.996,True
1,wait_min,excess_wait,0.954,True
2,late_min,late_min_adj,0.921,True
3,is_late,is_late_adj,0.855,True
4,late_min,is_late,0.711,True
5,late_min_adj,is_late_adj,0.694,True
6,late_min,CONFIRM_TO_DELIVER_DURATION,0.684,False
7,late_min,is_late_adj,0.660,True
8,late_min_adj,is_late,0.633,True
9,late_min_adj,severe,0.610,True


**33 · The plan doc's six named pairs**

,SUBTOTAL,TOTAL_ITEM_COUNT,hour_local,WAS_BATCHED,assign_lag_min,wait_min
r with late_min (ASAP orders),0.104,0.069,0.012,0.146,0.364,0.318


**candidate (weak):** on the plan doc's six pairs (ASAP orders, r with late_min): assignment lag 0.36, merchant wait 0.32, batched 0.15, subtotal 0.10, item count 0.07, hour 0.01 — dispatch and holds, not order size or time of day. In the wider sweep, travel-to-customer is the strongest non-definitional lateness correlate (r ≈ 0.5); rating correlates −0.24 to −0.26 with lateness.

### Findings summary
Human review done 2026-11-02; promotion tags below are the human's. Item numbers refer to the tables above. Nothing
here changed a metric. The deck work is the human's.

**PROMOTE-APPENDIX**
- **18** Doorlanta's data ends June 8: 600 orders over 8 days (~75/day) vs Dashattan's 13 (~46/day). Affects any "June 1–13" or
  per-day statement and makes the half-window check (23) uninformative. Also logged as cleaning-log row 18.

**PROMOTE-SLIDE**
- **26** Ratings collapse only beyond 15 min adjusted late (3.33 vs 4.45–4.69) — supports the severe-late threshold.
- **21** Dashattan bikes match cars on the customer leg (12.1 vs 12.1 min median; p90 24.1 vs 22.6) with a lower adjusted late
  rate — capacity, not speed, is the bike question.

**PROMOTE-Q3**
- **30** The ~17-min quote buffer ≈ p80 of estimate error (15.2 min); p90 would need 24 min.
- **15** Merchant wait scales with basket size (median 4.9 → 13.4 min; long holds 17% → 64%).

**APPENDIX**
- **28** Manual assignments: 25 orders, 52% late vs 13–17%, 2.5× dispatch lag, rating 3.8 vs 4.6.
- **25** Rating response bias in Dashattan: late orders rated 33% vs 47% (Doorlanta 53% / 53%) — flatters Dashattan's rating baseline.

**NOTE-ONLY**
- **22** Utilization concentration: KC 13 orders/active day, Elon 16; median Dasher 3.65 orders per active day, and only 18 of 54 reach the 6-per-Dasher-day sizing assumption (≈ 75th percentile).
- **14** Late rate and low-rating share rise with order value (plan-doc bands; $50+: 21% raw late, 17% low ratings; n = 95).
- **27** On on-time deliveries, long holds still dent ratings (4.71 → 4.49; low share 7.8% → 13.8%) — re-cut per the plan doc; worth a second look for the merchant metric. Orders with no logged wait are late 22.5% vs 6–14% when logged.
- **19** Lateness is lowest at the lunch peak and highest mid-afternoon; not peak-driven.
- **33** Plan doc's six pairs (ASAP): assignment lag 0.36, wait 0.32, batched 0.15, subtotal 0.10, items 0.07, hour 0.01. Travel-to-customer remains the strongest non-definitional correlate (r ≈ 0.5).
- **29** Fraud-flagged orders skew to big baskets (mean 8.4 items vs 3.4; $35 vs $25) — re-cut per the plan doc; no lateness or rating pattern.
- Nothing: 16 (subtotal by city), 17 (day of week), 20 (lag by hour), 23 (half-window: cannot assess), 24 (confirm speed),
  31 (vehicle consistency), 32 (IS_FIRST_DELIVERY).

In [70]:
# export_04_explore.xlsx: every sweep table, one sheet per item, title row first (optional export per CLAUDE.md)
sweep_sheets = {
    "item_14_subtotal_bands": [("14 · Late rate and rating by subtotal band (all orders)", t14, True)],
    "item_15_items_vs_wait": [("15 · Merchant wait by item-count band (all orders)", t15, True)],
    "item_16_subtotal_by_city": [("16 · Subtotal (USD) by city", t16, True)],
    "item_17_day_of_week": [("17 · Volume and late rate by day of week", t17, True)],
    "item_18_daily_trend": [("18 · Daily volume, late rate and median wait (Doorlanta ends June 8)", t18, True)],
    "item_19_late_by_hour": [("19 · Late rate by local hour (all orders)", t19, True)],
    "item_20_lag_by_hour": [("20 · Assignment lag (minutes) by local hour, ASAP", t20, True)],
    "item_21_bike_vs_car": [("21 · Dashattan: bicycle vs car", t21, True)],
    "item_22_utilization": [("22 · Utilization summary per market", t22, True), ("22 · Orders per active day, all Dashers", util.sort_values("orders_per_active_day", ascending=False), True)],
    "item_23_half_window": [("23 · Gate-passers: adjusted on-time per half of the window (ASAP)", t23, False)],
    "item_24_confirm_speed": [("24 · Assignment-to-confirmation seconds", t24, True)],
    "item_25_rating_response": [("25 · Rating response by late status", t25, True)],
    "item_26_rating_vs_late": [("26 · Rating by adjusted lateness band", t26, True)],
    "item_27_rating_vs_wait": [("27 · Rating by merchant-wait band", t27, True)],
    "item_28_manual_assign": [("28 · Manual vs automatic assignment", t28, True)],
    "item_29_fraud_rows": [("29 · Fraud-flagged deliveries", t29, False)],
    "item_30_estimate_error": [("30 · Delivery-time error distribution, ASAP (minutes)", t30, True)],
    "item_31_vehicle_consistency": [("31 · Orders by vehicle for multi-vehicle Dashers", t31, True)],
    "item_32_is_first": [("32 · City late rates with and without IS_FIRST_DELIVERY rows", t32, True)],
    "item_33_correlations": [("33 · Pairs with |r| > 0.2", t33, False)],
}
show("wrote export_04_explore.xlsx", write_workbook("export_04_explore.xlsx", {}, sweep_sheets))

**wrote export_04_explore.xlsx**

,data rows
item_14_subtotal_bands,4
item_15_items_vs_wait,5
item_16_subtotal_by_city,2
item_17_day_of_week,7
item_18_daily_trend,13
item_19_late_by_hour,17
item_20_lag_by_hour,16
item_21_bike_vs_car,2
item_22_utilization,56
item_23_half_window,20


## 9. Sensitivity

**What this section does:** re-runs the whole Dasher pipeline (row-level adjustment → ASAP filter → aggregates → shrink →
score → gates → tiers) under the alternatives listed in `CLAUDE.md` Phase 5 and reports, per market, how much of the
baseline Tier 1 survives: wait-forgiveness threshold 5 / 10 / 15 min · component weights (60, 20, 10, 10) and
(40, 40, 10, 10) · shrink k = 100 / 400 · scheduled orders included with a synthetic 17-min quote buffer.

**What it feeds:** the Appendix B stability claim ("Tier-1 cohort stable across thresholds & weights, at most one swap") and
the restructure note ("including scheduled orders leaves Tier 1 unchanged in Dashattan / 9 of 10 in Doorlanta"). Nothing
here changes the baseline scorecard; the baseline run must reproduce §5 exactly before any scenario is read.

### 9.1 Parameterised pipeline
One function, same definitions as §1 and §5, with the four knobs exposed. With scheduled orders included, their
lateness is measured against `quoted + 17 min` (the ASAP mean buffer) and the sample gate counts all included orders.

In [71]:
BASE = dict(wait_threshold=10, weights=(50, 30, 10, 10), k=200, include_scheduled=False)
SYNTHETIC_BUFFER_MIN = 17

def run_pipeline(wait_threshold=10, weights=(50, 30, 10, 10), k=200, include_scheduled=False):
    """Recompute Dasher scores and tiers under alternative parameters. Returns the scorecard-shaped frame."""
    w_rel, w_qual, w_cons, w_prof = weights
    r = df[["MARKET_NAME", "DASHER", "ASAP", "late_min", "wait_min"]].copy()
    r["excess"] = (r["wait_min"] - wait_threshold).clip(lower=0).fillna(0)
    if include_scheduled:
        r.loc[r["ASAP"] == 0, "late_min"] = r.loc[r["ASAP"] == 0, "late_min"] - SYNTHETIC_BUFFER_MIN
    else:
        r = r[r["ASAP"] == 1]
    r["late_adj"] = r["late_min"] - r["excess"]
    r["is_late_adj"] = r["late_adj"] > 0
    r["severe"] = r["late_adj"] > 15
    perf_s = r.groupby(["MARKET_NAME", "DASHER"]).agg(sample_n=("is_late_adj", "size"),
                                                      ontime_adj=("is_late_adj", lambda s: 1 - s.mean()),
                                                      severe_rate=("severe", "mean"))
    s = profile.join(perf_s, how="left")                       # lifetime profile and arrival share are parameter-free
    s["market_mean_star"] = s.index.get_level_values("MARKET_NAME").map(market_mean_star)
    n = s["lifetime_deliveries"]
    s["shrunk"] = (n * s["lifetime_star"] + k * s["market_mean_star"]) / (n + k)
    s["score"] = (w_rel * s["ontime_adj"] + w_qual * s["shrunk"] / 5
                  + w_cons * (1 - s["severe_rate"] / SEVERE_CAP).clip(lower=0) + w_prof * s["arrival_logged"])
    mk = s.index.get_level_values("MARKET_NAME")
    s["widget_capable"] = s["vehicles"].str.contains("car|motorcycle")
    s["passes"] = (s["widget_capable"] & (s["lifetime_deliveries"] >= mk.map(GATE_LIFETIME_MIN))
                   & (s["sample_n"].fillna(0) >= mk.map(GATE_ASAP_MIN)))
    s["tier"] = None
    for market, passers in s[s["passes"]].groupby(level="MARKET_NAME"):
        ranked = passers["score"].sort_values(ascending=False)
        s.loc[ranked.index, "tier"] = "Tier 2"
        s.loc[ranked.index[:TIER1_SIZE[market]], "tier"] = "Tier 1"
    return s.reset_index()

def tier1_sets(s):
    return {m: set(s.loc[(s["MARKET_NAME"] == m) & (s["tier"] == "Tier 1"), "DASHER"]) for m in TIER1_SIZE}

baseline_run = run_pipeline(**BASE)
baseline_t1 = tier1_sets(baseline_run)
reference_t1 = tier1_sets(scorecard)
assert baseline_t1 == reference_t1, (baseline_t1, reference_t1)
score_gap = (baseline_run.set_index(["MARKET_NAME", "DASHER"])["score"]
             - scorecard.set_index(["MARKET_NAME", "DASHER"])["selection_score"]).abs().max()
assert score_gap < 1e-9, score_gap
print("baseline run reproduces section 5 exactly: Tier 1 sets equal, max score gap", f"{score_gap:.1e}")
show("Baseline Tier 1", pd.DataFrame({m: sorted(v) + [""] * (10 - len(v)) for m, v in baseline_t1.items()}))

baseline run reproduces section 5 exactly: Tier 1 sets equal, max score gap 0.0e+00


**Baseline Tier 1**

,Dashattan,Doorlanta
0,Bill,Abhai
1,Carlee,Ajay
2,KC,Anisha
3,Samm,Cate
4,,Elon
5,,Hanna
6,,Hersh
7,,Jagar
8,,Josh
9,,Katelynn


### 9.2 Scenarios
For each scenario: how many of the baseline Tier 1 remain, and who swaps out / in. "Swaps" is the number of baseline
Tier-1 Dashers that drop out (equal to the number that come in, since tier sizes are fixed).

In [72]:
SCENARIOS = {
    "wait threshold 5 min": dict(BASE, wait_threshold=5),
    "wait threshold 15 min": dict(BASE, wait_threshold=15),
    "weights 60 / 20 / 10 / 10": dict(BASE, weights=(60, 20, 10, 10)),
    "weights 40 / 40 / 10 / 10": dict(BASE, weights=(40, 40, 10, 10)),
    "shrink k = 100": dict(BASE, k=100),
    "shrink k = 400": dict(BASE, k=400),
    "scheduled included (+17 min buffer)": dict(BASE, include_scheduled=True),
}
rows, scenario_runs = [], {}
for name, params in SCENARIOS.items():
    s = run_pipeline(**params)
    scenario_runs[name] = s
    t1 = tier1_sets(s)
    row = {"scenario": name}
    for m in TIER1_SIZE:
        out, inn = baseline_t1[m] - t1[m], t1[m] - baseline_t1[m]
        row[f"{m} kept"] = f"{len(baseline_t1[m] & t1[m])} of {TIER1_SIZE[m]}"
        row[f"{m} swaps"] = len(out)
        row[f"{m} out -> in"] = ", ".join(sorted(out)) + " -> " + ", ".join(sorted(inn)) if out else ""
    rows.append(row)
sensitivity = pd.DataFrame(rows).set_index("scenario")
show("Tier 1 overlap with baseline under each scenario", sensitivity)
sensitivity["max swaps"] = sensitivity[[f"{m} swaps" for m in TIER1_SIZE]].max(axis=1)

**Tier 1 overlap with baseline under each scenario**

,Dashattan kept,Dashattan swaps,Dashattan out -> in,Doorlanta kept,Doorlanta swaps,Doorlanta out -> in
scenario,,,,,,
wait threshold 5 min,4 of 4,0,,9 of 10,1,Cate -> Ash
wait threshold 15 min,4 of 4,0,,10 of 10,0,
weights 60 / 20 / 10 / 10,4 of 4,0,,10 of 10,0,
weights 40 / 40 / 10 / 10,4 of 4,0,,10 of 10,0,
shrink k = 100,4 of 4,0,,10 of 10,0,
shrink k = 400,4 of 4,0,,10 of 10,0,
scheduled included (+17 min buffer),4 of 4,0,,9 of 10,1,Hersh -> Ash


In [73]:
# the Dashers involved in any swap, with their score and rank under the scenario that moved them
detail = []
for name, s in scenario_runs.items():
    t1 = tier1_sets(s)
    for m in TIER1_SIZE:
        moved = (baseline_t1[m] ^ t1[m])
        if not moved:
            continue
        passers = s[(s["MARKET_NAME"] == m) & s["passes"]].sort_values("score", ascending=False).reset_index(drop=True)
        passers["rank"] = passers.index + 1
        for _, p in passers[passers["DASHER"].isin(moved)].iterrows():
            base = scorecard[(scorecard["MARKET_NAME"] == m) & (scorecard["DASHER"] == p["DASHER"])].iloc[0]
            detail.append({"scenario": name, "market": m, "dasher": p["DASHER"], "baseline score": round(base["selection_score"], 2),
                           "baseline tier": base["tier"], "scenario score": round(p["score"], 2), "scenario rank": int(p["rank"]),
                           "scenario tier": p["tier"]})
swap_detail = pd.DataFrame(detail)
show("Dashers that change tier under a scenario (empty = none)", swap_detail)

**Dashers that change tier under a scenario (empty = none)**

,scenario,market,dasher,baseline score,baseline tier,scenario score,scenario rank,scenario tier
0,wait threshold 5 min,Doorlanta,Ash,84.06,Tier 2,88.23,10,Tier 1
1,wait threshold 5 min,Doorlanta,Cate,86.62,Tier 1,87.33,11,Tier 2
2,scheduled included (+17 min buffer),Doorlanta,Ash,84.06,Tier 2,87.19,9,Tier 1
3,scheduled included (+17 min buffer),Doorlanta,Hersh,84.60,Tier 1,85.09,11,Tier 2


### 9.3 Claims check
Appendix B: "Tier-1 cohort stable across 5/10/15-min wait thresholds and weight shifts (60/20, 40/40): at most one swap
at the margin." Restructure note: "including scheduled orders leaves Tier 1 unchanged (Dashattan) / 9 of 10 (Doorlanta)."
k = 100 / 400 is in the Phase 5 spec with the same "identical or one swap" expectation.

In [74]:
wait_weight = sensitivity.loc[["wait threshold 5 min", "wait threshold 15 min", "weights 60 / 20 / 10 / 10", "weights 40 / 40 / 10 / 10"]]
k_rows = sensitivity.loc[["shrink k = 100", "shrink k = 400"]]
sched = sensitivity.loc["scheduled included (+17 min buffer)"]
register("Appendix B", "Tier 1 stable across wait thresholds & weights (max swaps <= 1)", 1, int((wait_weight["max swaps"] <= 1).all()), 0,
         "max swaps per scenario: " + ", ".join(f"{i}: {v}" for i, v in wait_weight["max swaps"].items()))
register("Appendix B", "Tier 1 stable across k = 100 / 400 (max swaps <= 1)", 1, int((k_rows["max swaps"] <= 1).all()), 0,
         "max swaps: " + ", ".join(f"{i}: {v}" for i, v in k_rows["max swaps"].items()))
register("Appendix B", "Scheduled included: Dashattan Tier 1 kept (of 4)", 4, 4 - sched["Dashattan swaps"], 0)
register("Appendix B", "Scheduled included: Doorlanta Tier 1 kept (of 10)", 9, 10 - sched["Doorlanta swaps"], 0, sched["Doorlanta out -> in"])
recon_tbl = pd.DataFrame(recon)
show("Phase 5 reconciliation rows", recon_tbl[recon_tbl["section"] == "Appendix B"])
print("Phase 5 acceptance:", "PASS" if (recon_tbl[recon_tbl["section"] == "Appendix B"]["match"] == "yes").all()
      else "CLAIMS DIFFER — see table")

**Phase 5 reconciliation rows**

,section,metric,expected,computed,tol,match,note
66,Appendix B,Tier 1 stable across wait thresholds & weights (max swaps <= 1),1.0,1.0,0.0,yes,"max swaps per scenario: wait threshold 5 min: 1, wait threshold 15 min: 0, weights 60 / 20 / 10 / 10: 0, weights 40 / 40 / 10 / 10: 0"
67,Appendix B,Tier 1 stable across k = 100 / 400 (max swaps <= 1),1.0,1.0,0.0,yes,"max swaps: shrink k = 100: 0, shrink k = 400: 0"
68,Appendix B,Scheduled included: Dashattan Tier 1 kept (of 4),4.0,4.0,0.0,yes,
69,Appendix B,Scheduled included: Doorlanta Tier 1 kept (of 10),9.0,9.0,0.0,yes,Hersh -> Ash


Phase 5 acceptance: PASS


### 9.4 Conclusion
The baseline run reproduces §5 exactly. Across the seven scenarios the Tier-1 cohort never changes by more than one Dasher per
market: Dashattan's four are identical in every scenario; Doorlanta swaps one only at wait threshold 5 (Cate → Ash) and with
scheduled orders included (Hersh → Ash) — both times Ash, the baseline 11th, edging past the 10th by about a point. Appendix B's
"at most one swap at the margin" and the "unchanged / 9 of 10" scheduled note both hold.

Registry status after Phase 5: every KEY_NUMBERS and Appendix B figure reconciles except two rows the human chose to keep
flagged: the dinner window (data says the peak runs 18–21, deck says 18–20; deck to widen its shading) and Josh's rank
(8 → 2 by score with raw vs adjusted on-time, deck said 7 → 2; deck to be corrected).

## 10. Slide-stat map

**What this does:** one row per number that appears on the deck, with the value the deck shows, the value this notebook computes,
the section that produces it, and whether they agree. Every value is pulled from the variables above, not retyped. Rows marked
*external / assumption* are not derivable from `data/` by design.

**What it feeds:** the final gate before submission — every slide stat traced to a cell.

In [75]:
def pct_of(series):
    return 100 * series.mean()

dash_t1 = scorecard[(scorecard["MARKET_NAME"] == "Dashattan") & (scorecard["tier"] == "Tier 1")].sort_values("selection_score", ascending=False)
door_t1 = scorecard[(scorecard["MARKET_NAME"] == "Doorlanta") & (scorecard["tier"] == "Tier 1")]["selection_score"]
travel_min = df["RESTAURANT_TO_CUSTOMER_DURATION"] / 60
capable = scorecard.groupby("MARKET_NAME")["widget_capable"].sum()
hold_share = hold_tbl.loc["share with hold > 10 min %", "value"]

# (deck stat, deck value, computed value, where, tolerance)   tolerance None = external / assumption, not checked
STATS = [
    ("Exec: widget-capable Dashers, Dashattan vs Doorlanta", "8 vs 30", f"{capable['Dashattan']} vs {capable['Doorlanta']}", "§5.2 scorecard", "exact"),
    ("Exec: share of Dashattan car deliveries on KC", "62%", f"{kc_share:.1f}%", "§3.2 kc_tbl", 0.5),
    ("Exec / Q1: Dashattan bicycle share of deliveries", "62%", f"{vehicle_mix.loc['bicycle', '% of deliveries']:.1f}%", "§3.2 vehicle_mix", 0.5),
    ("Exec: late deliveries with merchant hold > 10 min ('2 in 5')", "40%", f"{hold_share:.1f}%", "§4.3 hold_tbl", 0.5),
    ("2a: late deliveries without a long hold", "60%", f"{100 - hold_share:.1f}%", "§4.3 hold_tbl", 0.5),
    ("Exec / Q3: scheduled vs ASAP raw late rate", "27% vs 17%", f"{promise_tbl.loc['scheduled (ASAP=0)', 'late vs quoted %']:.1f}% vs {promise_tbl.loc['ASAP (ASAP=1)', 'late vs quoted %']:.1f}%", "§4.1 promise_tbl", 0.5),
    ("2a: ASAP late vs raw estimate", "56%", f"{promise_tbl.loc['ASAP (ASAP=1)', 'late vs raw estimate %']:.1f}%", "§4.1 promise_tbl", 0.5),
    ("2a / Q3: quote buffer, scheduled vs ASAP (min)", "0 vs ~17", f"{promise_tbl.loc['scheduled (ASAP=0)', 'mean quote buffer min']:.1f} vs {promise_tbl.loc['ASAP (ASAP=1)', 'mean quote buffer min']:.1f}", "§4.1 promise_tbl", 0.5),
    ("2b / Q3: Dashattan batched vs unbatched late (ASAP)", "33% vs 16%", f"{batch_asap.loc[('Dashattan', 'batched'), 'raw_late_pct']:.1f}% vs {batch_asap.loc[('Dashattan', 'unbatched'), 'raw_late_pct']:.1f}%", "§4.5 batch_asap", 0.5),
    ("2b / Q3: Doorlanta batched vs unbatched late (ASAP)", "16% vs 15%", f"{batch_asap.loc[('Doorlanta', 'batched'), 'raw_late_pct']:.1f}% vs {batch_asap.loc[('Doorlanta', 'unbatched'), 'raw_late_pct']:.1f}%", "§4.5 batch_asap", 0.5),
    ("2a: late rate, fastest -> slowest dispatch-lag quartile", "12% -> 26%", f"{lag_tbl.loc['Q1 fastest', 'raw_late_pct']:.1f}% -> {lag_tbl.loc['Q4 slowest', 'raw_late_pct']:.1f}%", "§4.4 lag_tbl", 0.5),
    ("Q1: uniform-strict qualifiers, Dashattan / Doorlanta", "1 / 8", f"{uniform_strict_counts['Dashattan']} / {uniform_strict_counts['Doorlanta']}", "§5.8", "exact"),
    ("Q1: city-calibrated gate passers, Dashattan / Doorlanta", "5 / 15", f"{gate_counts.loc['Dashattan', 'pass_all_gates']} / {gate_counts.loc['Doorlanta', 'pass_all_gates']}", "§2.2 gate_counts (re-derived §5.2)", "exact"),
    ("Q1: Tier 1 sizes", "4 / 10", f"{len(dash_t1)} / {len(door_t1)}", "§5.2", "exact"),
    ("Q1 / App C: Dashattan Tier 1 scores (KC, Samm, Carlee, Bill)", "92.5 / 84.6 / 80.4 / 80.1", " / ".join(f"{s:.1f}" for s in dash_t1["selection_score"]), "§5.2 scorecard", 0.05),
    ("Q1 / App C: Doorlanta Tier 1 score range", "86.6–97.4", f"{door_t1.min():.1f}–{door_t1.max():.1f}", "§5.2 scorecard", 0.05),
    ("Q1: KC fraud flags / rate", "2 (1.4% of 143)", f"{int(kc_tbl.loc['KC fraud flags', 'value'])} ({kc_tbl.loc['KC fraud flag rate %', 'value']:.1f}% of {int(kc_tbl.loc['KC observed deliveries (all)', 'value'])})", "§3.2 kc_tbl", 0.1),
    ("App B: Josh on-time raw -> adjusted", "80% -> 95%", f"{named.loc[('Doorlanta', 'Josh'), 'ontime_raw_pct']:.0f}% -> {named.loc[('Doorlanta', 'Josh'), 'ontime_adj_pct']:.0f}%", "§4.6 named", 0.5),
    ("App B: Josh rank", "7 -> 2 (deck to be corrected to 8 -> 2)", f"{score_reading[0]} -> {score_reading[1]}", "§5.6 rank_tbl", "flagged"),
    ("Q3: adjusted late rate baseline, ASAP (Dashattan / Doorlanta / pooled)", "13.4%", f"{baselines.loc['Dashattan', 'adjusted late % (ASAP)']:.1f}% / {baselines.loc['Doorlanta', 'adjusted late % (ASAP)']:.1f}% / {pct_of(asap['is_late_adj']):.1f}%", "§4.7 baselines; pooled from asap", "see note"),
    ("Q3: severe-late share baseline, ASAP (pooled)", "4.2%", f"{pct_of(asap['severe']):.1f}%", "pooled from asap (not in KEY_NUMBERS)", "see note"),
    ("Q3: average rating baseline (pooled)", "4.5", f"{df['DELIVERY_RATING'].mean():.2f}", "§3.1 city_summary (pooled from df)", 0.1),
    ("Q3: long-merchant-hold share baseline", "30%", f"{baselines.loc['Dashattan', 'long merchant hold % (wait > 10, all orders)']:.1f}% / {baselines.loc['Doorlanta', 'long merchant hold % (wait > 10, all orders)']:.1f}%", "§4.7 baselines", 0.5),
    ("Q3: low-rating share baseline", "~11–13%", f"{baselines.loc['Dashattan', 'low rating % (<= 3, of rated)']:.1f}% / {baselines.loc['Doorlanta', 'low rating % (<= 3, of rated)']:.1f}%", "§4.7 baselines", 0.5),
    ("Q2: satchel sizing chain", "1,000 -> 350 -> 58 -> 76", " -> ".join(str(int(v)) for v in sizing_funnel['rounded']), "§4.9 sizing_funnel", None),
    ("Q2: population density ratio", "24x (90,909 vs 3,731 /sq mi)", "external, not in data/", "§3.4", None),
    ("Q2: travel-to-customer leg p80 / p95 (pooled, min)", "p80 ~17 / p95 ~29", f"{travel_min.quantile(0.8):.1f} / {travel_min.quantile(0.95):.1f}", "pooled from df (p90 per city in §3.1)", 1.0),
    ("Q2: travel-to-customer p90 per city (min)", "~23 both", f"{city_summary.loc['Dashattan', 'p90 travel-to-customer min']:.1f} / {city_summary.loc['Doorlanta', 'p90 travel-to-customer min']:.1f}", "§3.1 city_summary", 1.0),
    ("Overview: Dashers per city", "24 / 30", f"{city_summary.loc['Dashattan', 'dashers']:.0f} / {city_summary.loc['Doorlanta', 'dashers']:.0f}", "§3.1 city_summary", "exact"),
    ("Overview: raw late rate per city", "19% / 16%", f"{city_summary.loc['Dashattan', 'raw late % (all orders)']:.1f}% / {city_summary.loc['Doorlanta', 'raw late % (all orders)']:.1f}%", "§3.1 city_summary", 0.5),
    ("Overview: median confirm-to-deliver (min)", "32 / 35", f"{city_summary.loc['Dashattan', 'median confirm-to-deliver min']:.1f} / {city_summary.loc['Doorlanta', 'median confirm-to-deliver min']:.1f}", "§3.1 city_summary", 0.5),
    ("Overview: average rating per city", "4.55 / 4.56", f"{city_summary.loc['Dashattan', 'avg delivery rating']:.2f} / {city_summary.loc['Doorlanta', 'avg delivery rating']:.2f}", "§3.1 city_summary", 0.01),
    ("Overview: median travel-to-customer (min)", "12.1 / 13.8", f"{city_summary.loc['Dashattan', 'median travel-to-customer min']:.1f} / {city_summary.loc['Doorlanta', 'median travel-to-customer min']:.1f}", "§3.1 city_summary", 0.1),
    ("Q3: scheduled share / median per Dasher / late with 17-min buffer", "6.2% / 2 / ~1%", f"{sched_tbl.loc['% of all orders', 'value']:.1f}% / {sched_tbl.loc['median scheduled orders per Dasher (Dashers with any)', 'value']:.0f} / {sched_tbl.loc['late % with synthetic 17-min buffer', 'value']:.1f}%", "§4.2 sched_tbl", 0.5),
    ("App A: junk rows removed", "6", f"{len(junk_rows)}", "§1.1 junk_rows", "exact"),
    ("App A: NUM_ON_TIME_DELIVERIES missing / implied lifetime on-time range", "41% / 1–77%", f"{ontime_tbl.loc['Both', '% missing']:.1f}% / {ontime_tbl.loc['Both', 'implied lifetime on-time % (min)']:.1f}–{ontime_tbl.loc['Both', 'implied lifetime on-time % (max)']:.1f}%", "§2.1 ontime_tbl", 0.5),
    ("App A: COMPOSITE_SCORE correlation with ratings", "r ~0.05–0.09", f"{corr_tbl.iloc[0, 0]:.2f} / {corr_tbl.iloc[1, 0]:.2f}", "§2.1 corr_tbl", 0.01),
    ("App B / Phase 5: Tier 1 max swaps across scenarios", "<= 1", f"{int(sensitivity['max swaps'].max())}", "§9.2 sensitivity", "exact"),
]

import re
def numbers(s):
    return [float(x) for x in re.findall(r"-?\d+\.?\d*", str(s).replace(",", ""))]

rows = []
for stat, deck, computed, where, tol in STATS:
    if tol is None:
        status = "external / assumption"
    elif tol == "flagged":
        status = "FLAGGED (human decision)"
    elif tol == "see note":
        status = "see note"
    else:
        d, c = numbers(deck), numbers(computed)
        if tol == "exact":
            status = "matches" if d == c else "DIFFERS"
        else:
            status = "matches" if len(d) == len(c) and all(abs(a - b) <= tol for a, b in zip(d, c)) else "DIFFERS"
    rows.append({"deck stat": stat, "deck value": deck, "notebook value": computed, "produced in": where, "status": status})
slide_map = pd.DataFrame(rows)
show("Slide-stat map: deck value vs notebook value", slide_map)
show("Rows that are not a clean match", slide_map[~slide_map["status"].isin(["matches", "external / assumption"])])

**Slide-stat map: deck value vs notebook value**

,deck stat,deck value,notebook value,produced in,status
0,"Exec: widget-capable Dashers, Dashattan vs Doorlanta",8 vs 30,8 vs 30,§5.2 scorecard,matches
1,Exec: share of Dashattan car deliveries on KC,62%,61.9%,§3.2 kc_tbl,matches
2,Exec / Q1: Dashattan bicycle share of deliveries,62%,61.5%,§3.2 vehicle_mix,matches
3,Exec: late deliveries with merchant hold > 10 min ('2 in 5'),40%,40.0%,§4.3 hold_tbl,matches
4,2a: late deliveries without a long hold,60%,60.0%,§4.3 hold_tbl,matches
5,Exec / Q3: scheduled vs ASAP raw late rate,27% vs 17%,27.0% vs 16.9%,§4.1 promise_tbl,matches
6,2a: ASAP late vs raw estimate,56%,55.6%,§4.1 promise_tbl,matches
7,"2a / Q3: quote buffer, scheduled vs ASAP (min)",0 vs ~17,0.0 vs 17.4,§4.1 promise_tbl,matches
8,2b / Q3: Dashattan batched vs unbatched late (ASAP),33% vs 16%,33.3% vs 15.8%,§4.5 batch_asap,matches
9,2b / Q3: Doorlanta batched vs unbatched late (ASAP),16% vs 15%,16.0% vs 14.9%,§4.5 batch_asap,matches


**Rows that are not a clean match**

,deck stat,deck value,notebook value,produced in,status
15,Q1 / App C: Doorlanta Tier 1 score range,86.6–97.4,84.6–97.4,§5.2 scorecard,DIFFERS
18,App B: Josh rank,7 -> 2 (deck to be corrected to 8 -> 2),8 -> 2,§5.6 rank_tbl,FLAGGED (human decision)
19,"Q3: adjusted late rate baseline, ASAP (Dashattan / Doorlanta / pooled)",13.4%,13.8% / 12.2% / 13.0%,§4.7 baselines; pooled from asap,see note
20,"Q3: severe-late share baseline, ASAP (pooled)",4.2%,4.4%,pooled from asap (not in KEY_NUMBERS),see note
22,Q3: long-merchant-hold share baseline,30%,30.2% / 29.7%,§4.7 baselines,DIFFERS
26,"Q2: travel-to-customer leg p80 / p95 (pooled, min)",p80 ~17 / p95 ~29,19.2 / 28.7,pooled from df (p90 per city in §3.1),DIFFERS
27,Q2: travel-to-customer p90 per city (min),~23 both,23.1 / 23.7,§3.1 city_summary,DIFFERS


**Notes on the 'see note' rows.** The deck's baseline strip shows "13.4%" for adjusted late rate: the notebook has 13.8% (Dashattan),
12.2% (Doorlanta) and the pooled ASAP figure printed above; "4.2%" is not a figure in KEY_NUMBERS — the pooled severe-late share is
printed beside it for comparison. Rows marked DIFFERS are deck numbers the notebook does not reproduce and are listed for the human.